**Environ 25 minutes d'exécution au lieu de plus de deux heures**

In [80]:
# Import de tout ce qui pourra être utile

import os
os.environ['CMDSTANPY_LOGGING'] = 'CRITICAL'
os.environ['CMDSTANPY_SHOW_PROGRESS'] = 'False'
from datetime import datetime, timedelta # Important pour manipuler des dates
import pandas as pd # Bibliothèque complète pour la manipulation de dataframes
import matplotlib.pyplot as plt # Pour tracer des graphiques
import numpy as np # Pour manipuler des tableaux, formatage différent de celui de pandas
import pmdarima as pmd # Pour effectuer des analyses de séries temporelles avec des modèles ARIMA
from sqlalchemy import create_engine # Pour créer une connexion à la BDD
from sqlalchemy.engine import URL # Pareil
from sqlalchemy import text # Pareil
import sqlalchemy # Pareil
import pyodbc # Pareil
from prophet import serialize
serialize.USE_STAN_BACKEND = 'pystan'
from prophet import Prophet
from pandas.api.types import is_datetime64_any_dtype as is_datetime # Pour manipuler des dates
from statsmodels.stats.diagnostic import acorr_ljungbox # Pour tester l'autocorrélation des résidus d'une série temporelle
from sklearn.metrics import mean_squared_error # Pour calculer l'erreur quadratique moyenne d'un modèle
from scipy import stats # Data science
import warnings # Permettra de cacher certains messages d'avertissement qui encombrent les outputs
from prophet.diagnostics import cross_validation, performance_metrics # Pour évaluer la robustesse et la performance des résultats d'un modèle
import contextlib 
import io
import logging
for lib in ['cmdstanpy', 'prophet'] :
    logger = logging.getLogger(lib)
    logger.handlers = []
    logger.setLevel(logging.CRITICAL)
import subprocess
import sys
import re
import time
import cmdstanpy
from scipy.stats import linregress
from statsmodels.tsa.stattools import adfuller
from scipy.fft import fft
from scipy.stats import entropy

In [ ]:
# Connexion pyodbc à SQL Server

conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=172.16.11.34;"
    "DATABASE=LogiProDev;"
    "UID=quentin;"
    "PWD={Barbidur1;SQL}"
)

conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

# Suppression systématique du contenu existant dans les deux tables
# Le but est de les remplir par la suite avec les noms et identifiants des graphiques de base ("historiques") puis avec
# les données de prévisions

print(f"Suppression du contenu de la table previsions_graphiques...")
cursor.execute(f"DELETE FROM previsions_graphiques")
print(f"Suppression du contenu de la table previsions_graphiques_noms...")
cursor.execute(f"DELETE FROM previsions_graphiques_noms")
conn.commit()

print("Normalement à ce stade les tables sont totalement vides")

Suppression du contenu de la table dbo.t_graphique_donnee...
Suppression du contenu de la table dbo.tr_graphique...
Normalement à ce stade les tables sont totalement vides


In [ ]:
# Remplissage de la table previsions_graphiques_noms : on y met les noms et identifiants des graphiques de données historiques

# C'est ce tableau qu'on modifie si on veut référencer ici une nouvelle série temporelle potentiellement exploitable
# pour la prévision, ou si on veut changer le numéro auquel correspond un graphique de données historiques (en cas de 
# suppression dans Superset d'un graphique historique qu'on recrée ensuite, son identifiant ne sera plus le même !)

libelles = [
    (146, "Action proposition et visite"),
    (147, "Nombre de transactions par négociateur"),
    (148, "Stock de demande par négociateur en surface"),
    (149, "Stock de demande par négociateur en nombre"),
    (150, "Stock d'offre par négociateur en surface"),
    (151, "Stock d'offre par négociateur en nombre"),
    (152, "Offres entrées"),
    (153, "Demandes entrées"),
    (154, "Nombre de plaintes et de désinscriptions"),
    (155, "Total emails envoyés"),
    (156, "Actions générales"),
    (157, "Evolution nombre de mails par opération"),
    (158, "Evolution activité globale"),
    (159, "Evolution activité de marché"),
    (160, "Analyse des erreurs"),
    (161, "Analyse des retours"),
    (162, "Origine des demandes"),
    (163, "Origine des offres"),
    (164, "Stock de mandats actifs"),
    (165, "Offres actives"),
    (166, "Demandes actives")
]

# Insertion SQL des noms et ids des graphiques historiques

for graphique_id, libelle in libelles :
    cursor.execute(
        "INSERT INTO previsions_graphiques_noms (graphique_id, gra_libelle) VALUES (?, ?)",
        (graphique_id, libelle)
    )
conn.commit()

print("Normalement à ce stade la table previsions_graphiques_noms contient les noms et identifiants des graphiques")
print("(La connexion est toujours ouverte)")

Normalement à ce stade la table dbo.tr_graphique contient les noms et identifiants des graphiques
(La connexion est toujours ouverte)


In [ ]:
# On commence par remonter les requêtes des datasets
# Ces requêtes sont des copié / collés des requêtes utilisées dans Superset pour créer les datasets utilisés pour générer
# les graphiques historiques. On remanipule ces datasets pour les mettre sous un format plus exploitable pour la prévision
# dans la suite de ce programme.

# Action proposition et visite
requete146 = """
SELECT
  A.groupe_id,
  CAST(A.act_date AS DATE) AS jour,
  AG.age_nom AS agence_nom,
  CASE A.action_type_id
    WHEN 1 THEN 'Visite'
    WHEN 2 THEN 'Visite Confrère'
    WHEN 3 THEN 'Proposition'
    WHEN 4 THEN 'Rdv Présentation'
    ELSE 'Autre'
  END AS type_action,
  COUNT(DISTINCT A.action_id) AS total_actions
FROM t_action A
  INNER JOIN t_negociateur N ON A.negociateur_id = N.negociateur_id
  INNER JOIN t_agence AG ON N.agence_id = AG.agence_id
  INNER JOIN tj_action_offre AO ON AO.action_id = A.action_id
  INNER JOIN t_offre O ON AO.offre_id = O.offre_id
  INNER JOIN tr_nature_offre NO ON NO.nature_offre_id = O.nature_offre_id AND NO.langue_id = 1
  INNER JOIN t_adresse AD ON AD.adresse_id = O.adresse_id
WHERE
  A.act_date BETWEEN DATEADD(YEAR, -17, GETDATE()) AND GETDATE()
  AND A.action_type_id IN (1, 2, 3, 4)
GROUP BY
  A.groupe_id,
  CAST(A.act_date AS DATE),
  AG.age_nom,
  CASE A.action_type_id
    WHEN 1 THEN 'Visite'
    WHEN 2 THEN 'Visite Confrère'
    WHEN 3 THEN 'Proposition'
    WHEN 4 THEN 'Rdv Présentation'
    ELSE 'Autre'
  END
"""

# Nombre de transactions par négociateur
requete147 = """ 
SELECT
    CAST(T.tra_date_signature AS DATE) AS jour,
    T.groupe_id,
    T.negociateur_id,
    N.neg_nom + ' ' + N.neg_prenom AS negociateur_nom,
    A.agence_id,
    A.age_nom AS agence_nom,
    COUNT(DISTINCT T.transaction_id) AS nb_transactions
FROM t_transaction T
INNER JOIN t_negociateur N ON T.negociateur_id = N.negociateur_id
INNER JOIN t_agence A ON N.agence_id = A.agence_id
WHERE
    T.tra_actif = 1
    AND T.tra_date_signature BETWEEN DATEADD(YEAR, -17, CAST(GETDATE() AS DATE)) AND CAST(GETDATE() AS DATE)
GROUP BY
    CAST(T.tra_date_signature AS DATE),
    T.groupe_id,
    T.negociateur_id,
    N.neg_nom,
    N.neg_prenom,
    A.agence_id,
    A.age_nom
"""

# Stock de demande par négociateur, en nombre et en surface
requete148et149 = """ 
SELECT 
  D.groupe_id,
  CAST(D.dem_date_creation AS DATE) AS jour,
  N.negociateur_id,
  N.agence_id,
  A.age_nom AS agence_nom,
  N.neg_nom + ' ' + N.neg_prenom AS negociateur_nom,
  COUNT(DISTINCT D.demande_id) AS nb_demandes,
  CAST(SUM(ISNULL(D.dem_recherche_surface_minimum1, 0)) AS BIGINT) AS surface_totale
FROM t_demande D
INNER JOIN tj_demande_negociateur DN ON DN.demande_id = D.demande_id
INNER JOIN t_negociateur N ON DN.negociateur_id = N.negociateur_id
INNER JOIN t_agence A ON N.agence_id = A.agence_id
WHERE 
  D.dem_date_creation BETWEEN DATEADD(YEAR, -17, GETDATE()) AND GETDATE()
GROUP BY 
  D.groupe_id,
  CAST(D.dem_date_creation AS DATE),
  N.negociateur_id,
  N.agence_id,
  A.age_nom,
  N.neg_nom,
  N.neg_prenom
"""

# Stock d'offre par négociateur, en nombre et en surface
requete150et151 = """ 
SELECT 
  O.groupe_id,
  CAST(COALESCE(O.off_date_creation, O.off_date_reactivation) AS DATE) AS jour,
  N.negociateur_id,
  N.agence_id,
  A.age_nom AS agence_nom,
  LTRIM(RTRIM(N.neg_nom)) + ' ' + LTRIM(RTRIM(N.neg_prenom)) AS negociateur_nom,
  COUNT(DISTINCT O.offre_id) AS nb_offres,
  CAST(ROUND(SUM(ISNULL(O.off_surface_totale, 0)), 0) AS BIGINT) AS surface_totale
FROM t_offre O
INNER JOIN tj_offre_negociateur ONEG ON ONEG.offre_id = O.offre_id
INNER JOIN t_negociateur N ON ONEG.negociateur_id = N.negociateur_id
INNER JOIN t_agence A ON N.agence_id = A.agence_id
WHERE 
  COALESCE(O.off_date_creation, O.off_date_reactivation) BETWEEN DATEADD(YEAR, -17, GETDATE()) AND GETDATE()
GROUP BY 
  O.groupe_id,
  CAST(COALESCE(O.off_date_creation, O.off_date_reactivation) AS DATE),
  N.negociateur_id,
  N.agence_id,
  A.age_nom,
  N.neg_nom,
  N.neg_prenom
"""

# Offres entrées 
requete152 = """ 
SELECT 
  CAST(
    CASE 
      WHEN O.off_date_creation BETWEEN DATEADD(YEAR, -17, CAST(GETDATE() AS DATE)) AND CAST(GETDATE() AS DATE)
        THEN O.off_date_creation
      ELSE O.off_date_reactivation
    END AS DATE
  ) AS jour,
  O.groupe_id,
  N.negociateur_id,
  A.agence_id,
  A.age_nom AS agence_nom,
  LTRIM(RTRIM(N.neg_nom)) + ' ' + LTRIM(RTRIM(N.neg_prenom)) AS negociateur_nom,
  COUNT(DISTINCT O.offre_id) AS nb_offres
FROM t_offre O
JOIN tj_offre_negociateur ONEG ON ONEG.offre_id = O.offre_id
JOIN t_negociateur N ON ONEG.negociateur_id = N.negociateur_id
JOIN t_agence A ON N.agence_id = A.agence_id
WHERE 
  (
    O.off_date_creation BETWEEN DATEADD(YEAR, -17, CAST(GETDATE() AS DATE)) AND CAST(GETDATE() AS DATE)
    OR O.off_date_reactivation BETWEEN DATEADD(YEAR, -17, CAST(GETDATE() AS DATE)) AND CAST(GETDATE() AS DATE)
  )
  AND N.neg_actif = 1
GROUP BY 
  CAST(
    CASE 
      WHEN O.off_date_creation BETWEEN DATEADD(YEAR, -17, CAST(GETDATE() AS DATE)) AND CAST(GETDATE() AS DATE)
        THEN O.off_date_creation
      ELSE O.off_date_reactivation
    END AS DATE
  ),
  O.groupe_id,
  N.negociateur_id,
  N.neg_nom,
  N.neg_prenom,
  A.agence_id,
  A.age_nom
"""

# Demandes entrées
requete153 = """ 
SELECT 
  CAST(D.dem_date_creation AS DATE) AS jour,
  D.groupe_id,
  N.negociateur_id,
  A.agence_id,
  A.age_nom AS agence_nom,
  LTRIM(RTRIM(N.neg_nom)) + ' ' + LTRIM(RTRIM(N.neg_prenom)) AS negociateur_nom,
  COUNT(DISTINCT D.demande_id) AS nb_demandes
FROM t_demande D
JOIN tj_demande_negociateur DNEG ON DNEG.demande_id = D.demande_id
JOIN t_negociateur N ON DNEG.negociateur_id = N.negociateur_id
JOIN t_agence A ON N.agence_id = A.agence_id
WHERE 
  D.dem_date_creation BETWEEN DATEADD(YEAR, -17, CAST(GETDATE() AS DATE)) AND CAST(GETDATE() AS DATE)
  AND N.neg_actif = 1
GROUP BY 
  CAST(D.dem_date_creation AS DATE),
  D.groupe_id,
  N.negociateur_id,
  N.neg_nom,
  N.neg_prenom,
  A.agence_id,
  A.age_nom
"""

# Nombre de plaintes et de désinscriptions ET Total emails envoyés
requete154et155 = """
SELECT
  CONVERT(date, OPE.op_date_envoi) AS date_envoi,
  OP.groupe_id,
  N.negociateur_id,
  LTRIM(RTRIM(N.neg_nom)) + ' ' + LTRIM(RTRIM(N.neg_prenom)) AS negociateur_nom,
  A.agence_id,
  A.age_nom AS agence_nom,
  COUNT(DISTINCT OPE.operation_personne_id) AS total_emails,
  COUNT(DISTINCT CASE WHEN ORS_plaintes.operation_personne_id IS NOT NULL THEN OPE.operation_personne_id END) AS nb_plaintes,
  COUNT(DISTINCT CASE WHEN ORS_desabo.operation_personne_id IS NOT NULL THEN OPE.operation_personne_id END) AS nb_desinscriptions,
  CASE 
    WHEN COUNT(DISTINCT OPE.operation_personne_id) = 0 THEN 0
    ELSE ROUND(
      CAST(COUNT(DISTINCT CASE WHEN ORS_plaintes.operation_personne_id IS NOT NULL THEN OPE.operation_personne_id END) AS FLOAT) 
      * 100.0 / COUNT(DISTINCT OPE.operation_personne_id), 
      2
    )
  END AS pct_plaintes,
  CASE 
    WHEN COUNT(DISTINCT OPE.operation_personne_id) = 0 THEN 0
    ELSE ROUND(
      CAST(COUNT(DISTINCT CASE WHEN ORS_desabo.operation_personne_id IS NOT NULL THEN OPE.operation_personne_id END) AS FLOAT) 
      * 100.0 / COUNT(DISTINCT OPE.operation_personne_id), 
      2
    )
  END AS pct_desinscriptions
FROM tj_operation_personne OPE
INNER JOIN t_operation_prospect OP ON OP.operation_prospect_id = OPE.operation_prospect_id
INNER JOIN t_negociateur N ON OP.negociateur_id = N.negociateur_id
LEFT JOIN t_agence A ON N.agence_id = A.agence_id
LEFT JOIN tj_operation_retour_SMTP ORS_plaintes 
  ON OPE.operation_personne_id = ORS_plaintes.operation_personne_id 
  AND ORS_plaintes.operation_type_retour_SMTP_id = 8
LEFT JOIN tj_operation_retour_SMTP ORS_desabo 
  ON OPE.operation_personne_id = ORS_desabo.operation_personne_id 
  AND ORS_desabo.operation_type_retour_SMTP_id = 9
WHERE
  OPE.op_date_envoi IS NOT NULL
  AND CONVERT(date, OPE.op_date_envoi) BETWEEN DATEADD(YEAR, -17, CAST(GETDATE() AS DATE)) AND CAST(GETDATE() AS DATE)
GROUP BY
  CONVERT(date, OPE.op_date_envoi),
  OP.groupe_id,
  N.negociateur_id,
  N.neg_nom,
  N.neg_prenom,
  A.agence_id,
  A.age_nom
"""

# Actions générales
requete156 = """ 
WITH top_actions AS (
  SELECT
    A.groupe_id,
    AT.at_libelle AS action_type,
    AT.action_type_id,
    A.negociateur_id,
    ROW_NUMBER() OVER (
      PARTITION BY A.groupe_id, A.negociateur_id
      ORDER BY COUNT(DISTINCT A.action_id) DESC
    ) AS rn
  FROM t_action A
  INNER JOIN tr_action_type AT
    ON AT.action_type_id = A.action_type_id
    AND AT.langue_id = 1
  WHERE
    A.act_date_creation BETWEEN DATEADD(YEAR, -17, GETDATE()) AND GETDATE()
    AND A.action_type_id IS NOT NULL
  GROUP BY
    A.groupe_id,
    AT.at_libelle,
    AT.action_type_id,
    A.negociateur_id
),
filtered_actions AS (
  SELECT
    A.groupe_id,
    CAST(A.act_date_creation AS DATE) AS jour,
    A.negociateur_id,
    TA.rn,
    COUNT(DISTINCT A.action_id) AS total_actions
  FROM t_action A
  INNER JOIN top_actions TA
    ON TA.action_type_id = A.action_type_id
    AND TA.groupe_id = A.groupe_id
    AND TA.negociateur_id = A.negociateur_id
    AND TA.rn <= 5
  WHERE
    A.act_date_creation BETWEEN DATEADD(YEAR, -17, GETDATE()) AND GETDATE()
  GROUP BY
    A.groupe_id,
    CAST(A.act_date_creation AS DATE),
    A.negociateur_id,
    TA.rn
)
SELECT
  fa.groupe_id,
  fa.jour,
  fa.negociateur_id,
  LTRIM(RTRIM(N.neg_nom)) + ' ' + LTRIM(RTRIM(N.neg_prenom)) AS negociateur_nom,
  N.agence_id,
  A.age_nom AS agence_nom,
  SUM(CASE WHEN fa.rn = 1 THEN fa.total_actions ELSE 0 END) AS action1,
  SUM(CASE WHEN fa.rn = 2 THEN fa.total_actions ELSE 0 END) AS action2,
  SUM(CASE WHEN fa.rn = 3 THEN fa.total_actions ELSE 0 END) AS action3,
  SUM(CASE WHEN fa.rn = 4 THEN fa.total_actions ELSE 0 END) AS action4,
  SUM(CASE WHEN fa.rn = 5 THEN fa.total_actions ELSE 0 END) AS action5
FROM filtered_actions fa
INNER JOIN t_negociateur N ON fa.negociateur_id = N.negociateur_id
LEFT JOIN t_agence A ON N.agence_id = A.agence_id
GROUP BY
  fa.groupe_id,
  fa.jour,
  fa.negociateur_id,
  N.neg_nom,
  N.neg_prenom,
  N.agence_id,
  A.age_nom
"""

# Evolution nombre de mails par opération
requete157 = """ 
SELECT
  CONVERT(date, OPE.op_date_envoi) AS date_envoi,
  OP.groupe_id,
  N.negociateur_id,
  LTRIM(RTRIM(N.neg_nom)) + ' ' + LTRIM(RTRIM(N.neg_prenom)) AS negociateur_nom,
  A.agence_id,
  A.age_nom AS agence_nom,
  COUNT(DISTINCT OPE.operation_personne_id) AS nb_total_emails,
  COUNT(DISTINCT OPE.operation_prospect_id) AS nb_operations,
  CASE 
    WHEN COUNT(DISTINCT OPE.operation_prospect_id) = 0 THEN 0
    ELSE COUNT(DISTINCT OPE.operation_personne_id) * 1.0 / COUNT(DISTINCT OPE.operation_prospect_id)
  END AS nb_emails_par_operation
FROM tj_operation_personne OPE
INNER JOIN t_operation_prospect OP 
  ON OP.operation_prospect_id = OPE.operation_prospect_id
INNER JOIN t_negociateur N 
  ON OP.negociateur_id = N.negociateur_id
LEFT JOIN t_agence A 
  ON N.agence_id = A.agence_id
WHERE
  OPE.op_date_envoi IS NOT NULL
  AND CONVERT(date, OPE.op_date_envoi) 
      BETWEEN DATEADD(YEAR, -17, CAST(GETDATE() AS DATE)) AND CAST(GETDATE() AS DATE)
GROUP BY
  CONVERT(date, OPE.op_date_envoi),
  OP.groupe_id,
  N.negociateur_id,
  N.neg_nom,
  N.neg_prenom,
  A.agence_id,
  A.age_nom
"""

# Evolution activité globale
requete158 = """ 
WITH offres AS (
  SELECT
    jour_data.jour,
    O.groupe_id,
    N.negociateur_id,
    LTRIM(RTRIM(N.neg_nom)) + ' ' + LTRIM(RTRIM(N.neg_prenom)) AS negociateur_nom,
    A.agence_id,
    A.age_nom AS agence_nom,
    COUNT(DISTINCT O.offre_id) AS nb_offres
  FROM t_offre O
  INNER JOIN tj_offre_negociateur ONEG ON ONEG.offre_id = O.offre_id
  INNER JOIN t_negociateur N ON ONEG.negociateur_id = N.negociateur_id
  LEFT JOIN t_agence A ON N.agence_id = A.agence_id

  CROSS APPLY (
    SELECT CAST(CASE 
                  WHEN O.off_date_creation BETWEEN DATEADD(YEAR, -17, GETDATE()) AND GETDATE()
                    THEN O.off_date_creation
                  ELSE O.off_date_reactivation 
                END AS DATE) AS jour
  ) AS jour_data

  WHERE
    (O.off_date_creation BETWEEN DATEADD(YEAR, -17, GETDATE()) AND GETDATE()
     OR O.off_date_reactivation BETWEEN DATEADD(YEAR, -17, GETDATE()) AND GETDATE())
    
  GROUP BY
    jour_data.jour,
    O.groupe_id,
    N.negociateur_id,
    N.neg_nom,
    N.neg_prenom,
    A.agence_id,
    A.age_nom
),

demandes AS (
  SELECT
    CAST(D.dem_date_creation AS DATE) AS jour,
    D.groupe_id,
    N.negociateur_id,
    LTRIM(RTRIM(N.neg_nom)) + ' ' + LTRIM(RTRIM(N.neg_prenom)) AS negociateur_nom,
    A.agence_id,
    A.age_nom AS agence_nom,
    COUNT(DISTINCT D.demande_id) AS nb_demandes
  FROM t_demande D
  INNER JOIN tj_demande_negociateur DN ON DN.demande_id = D.demande_id
  INNER JOIN t_negociateur N ON DN.negociateur_id = N.negociateur_id
  LEFT JOIN t_agence A ON N.agence_id = A.agence_id
  WHERE D.dem_date_creation BETWEEN DATEADD(YEAR, -17, GETDATE()) AND GETDATE()
  GROUP BY
    CAST(D.dem_date_creation AS DATE),
    D.groupe_id,
    N.negociateur_id,
    N.neg_nom,
    N.neg_prenom,
    A.agence_id,
    A.age_nom
),

mandats AS (
  SELECT
    CAST(M.man_date_creation AS DATE) AS jour,
    A.groupe_id,
    N.negociateur_id,
    LTRIM(RTRIM(N.neg_nom)) + ' ' + LTRIM(RTRIM(N.neg_prenom)) AS negociateur_nom,
    A.agence_id,
    A.age_nom AS agence_nom,
    COUNT(DISTINCT M.mandat_id) AS nb_mandats
  FROM t_mandat M
  INNER JOIN t_negociateur N ON M.negociateur_id = N.negociateur_id
  LEFT JOIN t_agence A ON A.agence_id = M.agence_id
  WHERE M.man_date_creation BETWEEN DATEADD(YEAR, -17, GETDATE()) AND GETDATE()
  GROUP BY
    CAST(M.man_date_creation AS DATE),
    A.groupe_id,
    N.negociateur_id,
    N.neg_nom,
    N.neg_prenom,
    A.agence_id,
    A.age_nom
),

cte_union AS (
  SELECT
    COALESCE(o.jour, d.jour, m.jour) AS jour,
    COALESCE(o.groupe_id, d.groupe_id, m.groupe_id) AS groupe_id,
    COALESCE(o.negociateur_id, d.negociateur_id, m.negociateur_id) AS negociateur_id,
    COALESCE(o.negociateur_nom, d.negociateur_nom, m.negociateur_nom) AS negociateur_nom,
    COALESCE(o.agence_id, d.agence_id, m.agence_id) AS agence_id,
    COALESCE(o.agence_nom, d.agence_nom, m.agence_nom) AS agence_nom,
    ISNULL(o.nb_offres, 0) AS nb_offres,
    ISNULL(d.nb_demandes, 0) AS nb_demandes,
    ISNULL(m.nb_mandats, 0) AS nb_mandats
  FROM offres o
  FULL OUTER JOIN demandes d ON o.jour = d.jour AND o.groupe_id = d.groupe_id AND o.negociateur_id = d.negociateur_id
  FULL OUTER JOIN mandats m ON COALESCE(o.jour, d.jour) = m.jour AND COALESCE(o.groupe_id, d.groupe_id) = m.groupe_id AND COALESCE(o.negociateur_id, d.negociateur_id) = m.negociateur_id
)

SELECT
  jour,
  groupe_id,
  negociateur_id,
  negociateur_nom,
  agence_id,
  agence_nom,
  SUM(nb_offres) AS nb_offres,
  SUM(nb_demandes) AS nb_demandes,
  SUM(nb_mandats) AS nb_mandats
FROM cte_union
WHERE
  groupe_id IN (38)
  AND jour IS NOT NULL
GROUP BY
  jour,
  groupe_id,
  negociateur_id,
  negociateur_nom,
  agence_id,
  agence_nom
"""

# Evolution activité de marché
requete159 = """ 
-- Étape 1 : Prétraitement pour les offres
WITH cte_base_offres AS (
  SELECT
    O.offre_id,
    CAST(CASE 
           WHEN O.off_date_creation BETWEEN DATEADD(YEAR, -17, CAST(GETDATE() AS DATE)) AND CAST(GETDATE() AS DATE)
             THEN O.off_date_creation
           ELSE O.off_date_reactivation
         END AS DATE) AS jour,
    O.groupe_id,
    N.negociateur_id,
    LTRIM(RTRIM(N.neg_nom)) + ' ' + LTRIM(RTRIM(N.neg_prenom)) AS negociateur_nom,
    A.agence_id,
    A.age_nom AS agence_nom
  FROM t_offre O
  INNER JOIN tj_offre_negociateur ONEG ON ONEG.offre_id = O.offre_id
  INNER JOIN t_negociateur N ON ONEG.negociateur_id = N.negociateur_id
  LEFT JOIN t_agence A ON N.agence_id = A.agence_id
  INNER JOIN tr_nature_offre NO ON O.nature_offre_id = NO.nature_offre_id AND NO.langue_id = 1
  WHERE
    (O.off_date_creation BETWEEN DATEADD(YEAR, -17, GETDATE()) AND GETDATE())
    OR (O.off_date_reactivation BETWEEN DATEADD(YEAR, -17, GETDATE()) AND GETDATE())
),

cte_offres AS (
  SELECT
    jour,
    groupe_id,
    negociateur_id,
    negociateur_nom,
    agence_id,
    agence_nom,
    COUNT(DISTINCT offre_id) AS nb_offres
  FROM cte_base_offres
  GROUP BY
    jour,
    groupe_id,
    negociateur_id,
    negociateur_nom,
    agence_id,
    agence_nom
),

cte_demandes AS (
  SELECT
    CAST(D.dem_date_creation AS DATE) AS jour,
    D.groupe_id,
    N.negociateur_id,
    LTRIM(RTRIM(N.neg_nom)) + ' ' + LTRIM(RTRIM(N.neg_prenom)) AS negociateur_nom,
    A.agence_id,
    A.age_nom AS agence_nom,
    COUNT(DISTINCT D.demande_id) AS nb_demandes
  FROM t_demande D
  INNER JOIN tj_demande_negociateur DN ON DN.demande_id = D.demande_id
  INNER JOIN t_negociateur N ON DN.negociateur_id = N.negociateur_id
  LEFT JOIN t_agence A ON N.agence_id = A.agence_id
  WHERE D.dem_date_creation BETWEEN DATEADD(YEAR, -17, GETDATE()) AND GETDATE()
  GROUP BY
    CAST(D.dem_date_creation AS DATE),
    D.groupe_id,
    N.negociateur_id,
    N.neg_nom,
    N.neg_prenom,
    A.agence_id,
    A.age_nom
),

cte_mandats AS (
  SELECT
    CAST(M.man_date_creation AS DATE) AS jour,
    A.groupe_id,
    N.negociateur_id,
    LTRIM(RTRIM(N.neg_nom)) + ' ' + LTRIM(RTRIM(N.neg_prenom)) AS negociateur_nom,
    A.agence_id,
    A.age_nom AS agence_nom,
    COUNT(DISTINCT M.mandat_id) AS nb_mandats
  FROM t_mandat M
  INNER JOIN tj_mandat_offre MO ON MO.mandat_id = M.mandat_id
  INNER JOIN t_offre O ON O.offre_id = MO.offre_id
  INNER JOIN tr_nature_offre NO ON O.nature_offre_id = NO.nature_offre_id AND NO.langue_id = 1
  INNER JOIN t_negociateur N ON M.negociateur_id = N.negociateur_id
  LEFT JOIN t_agence A ON A.agence_id = M.agence_id
  WHERE M.man_date_creation BETWEEN DATEADD(YEAR, -17, GETDATE()) AND GETDATE()
  GROUP BY
    CAST(M.man_date_creation AS DATE),
    A.groupe_id,
    N.negociateur_id,
    N.neg_nom,
    N.neg_prenom,
    A.agence_id,
    A.age_nom
),

activite_marche_journaliere AS (
  SELECT
    COALESCE(o.jour, d.jour, m.jour) AS jour,
    COALESCE(o.groupe_id, d.groupe_id, m.groupe_id) AS groupe_id,
    COALESCE(o.negociateur_id, d.negociateur_id, m.negociateur_id) AS negociateur_id,
    COALESCE(o.negociateur_nom, d.negociateur_nom, m.negociateur_nom) AS negociateur_nom,
    COALESCE(o.agence_id, d.agence_id, m.agence_id) AS agence_id,
    COALESCE(o.agence_nom, d.agence_nom, m.agence_nom) AS agence_nom,
    ISNULL(o.nb_offres, 0) AS nb_offres,
    ISNULL(d.nb_demandes, 0) AS nb_demandes,
    ISNULL(m.nb_mandats, 0) AS nb_mandats
  FROM cte_offres o
  FULL OUTER JOIN cte_demandes d ON o.jour = d.jour AND o.groupe_id = d.groupe_id AND o.negociateur_id = d.negociateur_id
  FULL OUTER JOIN cte_mandats m ON COALESCE(o.jour, d.jour) = m.jour AND COALESCE(o.groupe_id, d.groupe_id) = m.groupe_id AND COALESCE(o.negociateur_id, d.negociateur_id) = m.negociateur_id
)

SELECT
  jour,
  groupe_id,
  negociateur_id,
  negociateur_nom,
  agence_id,
  agence_nom,
  SUM(nb_offres) AS nb_offres,
  SUM(nb_demandes) AS nb_demandes,
  SUM(nb_mandats) AS nb_mandats
FROM activite_marche_journaliere
WHERE jour IS NOT NULL
GROUP BY
  jour,
  groupe_id,
  negociateur_id,
  negociateur_nom,
  agence_id,
  agence_nom
"""

# Analyse des erreurs
requete160 = """ 
WITH erreurs AS (

  SELECT 
    CAST(OPE.op_date_envoi AS DATE) AS jour,
    'Rejeté' AS type_erreur, 
    OP.groupe_id, 
    N.negociateur_id,
    LTRIM(RTRIM(N.neg_nom)) + ' ' + LTRIM(RTRIM(N.neg_prenom)) AS negociateur_nom,
    A.agence_id,
    A.age_nom AS agence_nom,
    COUNT(DISTINCT ORS_rejete.operation_personne_id) AS nb
  FROM tj_operation_personne OPE
  INNER JOIN t_operation_prospect OP ON OP.operation_prospect_id = OPE.operation_prospect_id
  INNER JOIN t_negociateur N ON OP.negociateur_id = N.negociateur_id
  LEFT JOIN t_agence A ON N.agence_id = A.agence_id
  LEFT JOIN tj_operation_retour_SMTP ORS_rejete
    ON OPE.operation_personne_id = ORS_rejete.operation_personne_id
    AND ORS_rejete.operation_type_retour_SMTP_id = 2
  WHERE 
    OPE.op_date_envoi IS NOT NULL
    AND CONVERT(date, OPE.op_date_envoi) BETWEEN DATEADD(YEAR, -17, CAST(GETDATE() AS DATE)) AND CAST(GETDATE() AS DATE)
  GROUP BY 
    CAST(OPE.op_date_envoi AS DATE),
    OP.groupe_id,
    N.negociateur_id,
    N.neg_nom,
    N.neg_prenom,
    A.agence_id,
    A.age_nom

  UNION ALL

  SELECT 
    CAST(OPE.op_date_envoi AS DATE) AS jour,
    'Erreur' AS type_erreur, 
    OP.groupe_id, 
    N.negociateur_id,
    LTRIM(RTRIM(N.neg_nom)) + ' ' + LTRIM(RTRIM(N.neg_prenom)) AS negociateur_nom,
    A.agence_id,
    A.age_nom AS agence_nom,
    COUNT(DISTINCT ORS_erreur.operation_personne_id) AS nb
  FROM tj_operation_personne OPE
  INNER JOIN t_operation_prospect OP ON OP.operation_prospect_id = OPE.operation_prospect_id
  INNER JOIN t_negociateur N ON OP.negociateur_id = N.negociateur_id
  LEFT JOIN t_agence A ON N.agence_id = A.agence_id
  LEFT JOIN tj_operation_retour_SMTP ORS_erreur
    ON OPE.operation_personne_id = ORS_erreur.operation_personne_id
    AND ORS_erreur.operation_type_retour_SMTP_id = 1
  WHERE 
    OPE.op_date_envoi IS NOT NULL
    AND CONVERT(date, OPE.op_date_envoi) BETWEEN DATEADD(YEAR, -17, CAST(GETDATE() AS DATE)) AND CAST(GETDATE() AS DATE)
  GROUP BY 
    CAST(OPE.op_date_envoi AS DATE),
    OP.groupe_id,
    N.negociateur_id,
    N.neg_nom,
    N.neg_prenom,
    A.agence_id,
    A.age_nom

  UNION ALL

  SELECT 
    CAST(OPE.op_date_envoi AS DATE) AS jour,
    'NPAI' AS type_erreur, 
    OP.groupe_id, 
    N.negociateur_id,
    LTRIM(RTRIM(N.neg_nom)) + ' ' + LTRIM(RTRIM(N.neg_prenom)) AS negociateur_nom,
    A.agence_id,
    A.age_nom AS agence_nom,
    COUNT(DISTINCT ORS_hardbounce.operation_personne_id) AS nb
  FROM tj_operation_personne OPE
  INNER JOIN t_operation_prospect OP ON OP.operation_prospect_id = OPE.operation_prospect_id
  INNER JOIN t_negociateur N ON OP.negociateur_id = N.negociateur_id
  LEFT JOIN t_agence A ON N.agence_id = A.agence_id
  LEFT JOIN tj_operation_retour_SMTP ORS_hardbounce
    ON OPE.operation_personne_id = ORS_hardbounce.operation_personne_id
    AND ORS_hardbounce.operation_type_retour_SMTP_id = 3
  WHERE 
    OPE.op_date_envoi IS NOT NULL
    AND CONVERT(date, OPE.op_date_envoi) BETWEEN DATEADD(YEAR, -17, CAST(GETDATE() AS DATE)) AND CAST(GETDATE() AS DATE)
  GROUP BY 
    CAST(OPE.op_date_envoi AS DATE),
    OP.groupe_id,
    N.negociateur_id,
    N.neg_nom,
    N.neg_prenom,
    A.agence_id,
    A.age_nom
)

SELECT 
  jour,
  type_erreur, 
  groupe_id, 
  negociateur_id,
  negociateur_nom,
  agence_id,
  agence_nom,
  nb
FROM erreurs
"""

# Analyse des retours
requete161 = """ 
WITH base AS (
  SELECT
    CONVERT(date, OPE.op_date_envoi) AS jour,
    OP.groupe_id,
    N.negociateur_id,
    LTRIM(RTRIM(N.neg_nom)) + ' ' + LTRIM(RTRIM(N.neg_prenom)) AS negociateur_nom,
    A.agence_id,
    A.age_nom AS agence_nom,
    OPE.operation_personne_id,
    MAX(CASE WHEN ORS.operation_type_retour_SMTP_id = 5 THEN 1 ELSE 0 END) AS delivre,
    MAX(CASE WHEN ORS.operation_type_retour_SMTP_id = 6 THEN 1 ELSE 0 END) AS ouvert,
    MAX(CASE WHEN ORS.operation_type_retour_SMTP_id = 7 THEN 1 ELSE 0 END) AS clique
  FROM tj_operation_personne OPE
  INNER JOIN t_operation_prospect OP ON OP.operation_prospect_id = OPE.operation_prospect_id
  INNER JOIN t_negociateur N ON OP.negociateur_id = N.negociateur_id
  LEFT JOIN t_agence A ON N.agence_id = A.agence_id
  LEFT JOIN tj_operation_retour_SMTP ORS ON OPE.operation_personne_id = ORS.operation_personne_id
  WHERE 
    OPE.op_date_envoi IS NOT NULL
    AND CONVERT(date, OPE.op_date_envoi) 
        BETWEEN DATEADD(YEAR, -17, CAST(GETDATE() AS DATE)) 
        AND CAST(GETDATE() AS DATE)
  GROUP BY
    CONVERT(date, OPE.op_date_envoi),
    OP.groupe_id,
    N.negociateur_id,
    N.neg_nom,
    N.neg_prenom,
    A.agence_id,
    A.age_nom,
    OPE.operation_personne_id
),

catégories AS (
  SELECT
    jour,
    groupe_id,
    negociateur_id,
    negociateur_nom,
    agence_id,
    agence_nom,
    CASE
      WHEN clique = 1 THEN 'Cliqués'
      WHEN ouvert = 1 THEN 'Ouverts'
      WHEN delivre = 1 THEN 'Délivrés'
      ELSE 'Rejetés'
    END AS type_retour
  FROM base
)

SELECT
  jour,
  groupe_id,
  negociateur_id,
  negociateur_nom,
  agence_id,
  agence_nom,
  type_retour,
  COUNT(*) AS nb
FROM catégories
GROUP BY 
  jour,
  groupe_id,
  negociateur_id,
  negociateur_nom,
  agence_id,
  agence_nom,
  type_retour
"""

# Pas très intéressant a priori de prévoir les origines d'offres et de demandes, surtout que par origine, il y en a trop peu

# Stock de mandats actifs
requete164 = """ 
WITH mois_actifs AS (
  -- Génère les mois entre début et fin du filtre; Superset gère cela via le time grain
  SELECT
    DATEADD(MONTH, DATEDIFF(MONTH, 0, CAST(GETDATE() AS DATE)) - n, 0) AS mois
  FROM (
    SELECT TOP (204) ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) - 1 AS n
    FROM master..spt_values
  ) numbers
),

mandats_base AS (
  SELECT
    A.groupe_id,
    N.negociateur_id,
    LTRIM(RTRIM(N.neg_nom)) + ' ' + LTRIM(RTRIM(N.neg_prenom)) AS negociateur_nom,
    A.agence_id,
    A.age_nom AS agence_nom,
    M.mandat_type_id,
    CASE 
      WHEN MT.man_location_vente = 1 THEN MT.man_libelle ELSE 'Demande'
    END AS mandat_type,
    M.man_date_debut,
    M.man_date_fin
  FROM t_mandat M
  JOIN t_negociateur N ON M.negociateur_id = N.negociateur_id
  JOIN t_agence A ON A.agence_id = M.agence_id
  JOIN tr_mandat_type MT 
    ON MT.mandat_type_id = M.mandat_type_id AND MT.langue_id = 1
  WHERE M.man_date_fin >= DATEADD(YEAR, -17, CAST(GETDATE() AS DATE))
),

activite_mandats AS (
  SELECT
    ma.mois,
    mb.groupe_id,
    mb.negociateur_id,
    mb.negociateur_nom,
    mb.agence_id,
    mb.agence_nom,
    mb.mandat_type,
    COUNT(DISTINCT mb.mandat_type_id) AS cpt_mandats
  FROM mois_actifs ma
  JOIN mandats_base mb
    ON mb.man_date_debut <= DATEADD(MONTH, 1, ma.mois)  -- début avant fin du mois
    AND mb.man_date_fin >= ma.mois                      -- fin après début du mois
  GROUP BY
    ma.mois,
    mb.groupe_id,
    mb.negociateur_id,
    mb.negociateur_nom,
    mb.agence_id,
    mb.agence_nom,
    mb.mandat_type
)

SELECT
  mois,
  groupe_id,
  negociateur_id,
  negociateur_nom,
  agence_id,
  agence_nom,
  mandat_type,
  SUM(cpt_mandats) AS total_actifs
FROM activite_mandats
GROUP BY
  mois,
  groupe_id,
  negociateur_id,
  negociateur_nom,
  agence_id,
  agence_nom,
  mandat_type
"""

# Offres actives et Demandes actives ne feront pas l'objet de prévisions pour l'instant, il s'agit d'une vue en t du stock
# d'offres et de demandes
# C'est à dire que les "graphiques" 165 et 166 ne sont pas ici ! (ce sont des "big numbers", pas vraiment des graphiques
# d'alleurs)

# On ne fait remonter que la liste des groupes actifs, pas besoin d'encombrer ce programme déjà long à l'exécution par 
# les données de groupes qui n'utilisent plus LGP

requete_groupes = """select groupe_id from dbo.t_groupe where gro_actif = 1"""

In [84]:
# On remonte les datasets nécessaires pour faire les prévisions (compter deux minutes d'exécution)

dataset146 = pd.read_sql_query(requete146, conn) # Action proposition et visite
dataset147 = pd.read_sql_query(requete147, conn) # Nombre de transactions par négociateur
dataset148et149 = pd.read_sql_query(requete148et149, conn) # Stock de demande par négociateur
dataset150et151 = pd.read_sql_query(requete150et151, conn) # Stock d'offre par négociateur
dataset152 = pd.read_sql_query(requete152, conn) # Offres entrées
dataset153 = pd.read_sql_query(requete153, conn) # Demandes entrées
dataset154et155 = pd.read_sql_query(requete154et155, conn) # Nombre de plaintes et de désinscriptions
dataset156 = pd.read_sql_query(requete156, conn) # Actions générales
dataset157 = pd.read_sql_query(requete157, conn) # Evolution nombre de mails par opération
dataset158 = pd.read_sql_query(requete158, conn) # Evolution activité globale
dataset159 = pd.read_sql_query(requete159, conn) # Evolution activité de marché
dataset160 = pd.read_sql_query(requete160, conn) # Analyse des erreurs
dataset161 = pd.read_sql_query(requete161, conn) # Analyse des retours
dataset164 = pd.read_sql_query(requete164, conn) # Stock de mandats actifs

groupes = pd.read_sql_query(requete_groupes,conn) # df contenant tous les id de groupes actifs

conn.commit()
cursor.close()
conn.close()

# On ferme la connexion pyodbc à la BDD parce que le programme est long à exécuter, on n'en aura plus besoin avant dans
# environ une heure et demie

C:\Users\quent\AppData\Local\Temp\ipykernel_16388\1573457.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dataset146 = pd.read_sql_query(requete146, conn) # Action proposition et visite
C:\Users\quent\AppData\Local\Temp\ipykernel_16388\1573457.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dataset147 = pd.read_sql_query(requete147, conn) # Nombre de transactions par négociateur
C:\Users\quent\AppData\Local\Temp\ipykernel_16388\1573457.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dataset148et149 = pd.read_sql

In [85]:
# On trie les groupes par identifiant

groupes = groupes.sort_values(by='groupe_id', ascending=True)
groupes = groupes['groupe_id'].tolist() # on les transforme en liste au cas où on aurait besoin de manipuler cette liste,
# objet un peu plus manipulable

In [86]:
# On commence par diviser en plusieurs datasets les datasets servant à afficher plusieurs séries temporelles 
# sur un même graphique
# Ca concerne les datasets 146, 156, 158, 159, 160, 161, 164

# Action proposition et visite : on sépare les trois types d'actions 
dataset146visite = dataset146[dataset146['type_action'] == 'Visite']
dataset146rdv = dataset146[dataset146['type_action'] == 'Rdv Présentation']
dataset146proposition = dataset146[dataset146['type_action'] == 'Proposition']

dataset148 = dataset148et149 # Demande en surface
if 'nb_demandes' in dataset148.columns :
    dataset148 = dataset148.drop('nb_demandes', axis=1)

dataset149 = dataset148et149 # Demande en nombre
if 'surface_totale' in dataset149.columns :
    dataset149 = dataset149.drop('surface_totale', axis=1)

dataset150 = dataset150et151 # Offre en surface
if 'nb_offres' in dataset150.columns : 
    dataset150 = dataset150.drop('nb_offres', axis=1)

dataset151 = dataset150et151 # Offre en nombre
if 'surface_totale' in dataset151.columns : 
    dataset151 = dataset151.drop('surface_totale', axis=1)

dataset154 = dataset154et155 # Nombre de plaintes et de désinscriptions : ON AGREGE LES DEUX
dataset154['plaintes_desinscriptions'] = dataset154['nb_plaintes'] + dataset154['nb_desinscriptions']
if 'total_emails' in dataset154.columns : 
    dataset154 = dataset154.drop('total_emails', axis=1)

dataset155 = dataset154et155 # Total emails envoyés
if 'plaintes_desinscriptions' in dataset155.columns : 
    dataset155 = dataset155.drop('plaintes_desinscriptions', axis=1)

# Actions générales
dataset156action1 = dataset156.drop(['action2', 'action3', 'action4', 'action5'], axis=1)
dataset156action2 = dataset156.drop(['action1', 'action3', 'action4', 'action5'], axis=1)
dataset156action3 = dataset156.drop(['action1', 'action2', 'action4', 'action5'], axis=1)
dataset156action4 = dataset156.drop(['action1', 'action2', 'action3', 'action5'], axis=1)
dataset156action5 = dataset156.drop(['action1', 'action2', 'action3', 'action4'], axis=1)

# Evolution activité globale
dataset158offres = dataset158.drop(['nb_demandes', 'nb_mandats'], axis=1)
dataset158demandes = dataset158.drop(['nb_offres', 'nb_mandats'], axis=1)
dataset158mandats = dataset158.drop(['nb_offres', 'nb_demandes'], axis=1)

# Evolution activité de marché
dataset159offres = dataset159.drop(['nb_demandes', 'nb_mandats'], axis=1)
dataset159demandes = dataset159.drop(['nb_offres', 'nb_mandats'], axis=1)
dataset159mandats = dataset159.drop(['nb_offres', 'nb_demandes'], axis=1)

# Analyse des erreurs (erreur tout court est à 0 partout donc on l'oublie)
dataset160rejete = dataset160[dataset160['type_erreur'] == 'Rejeté']
dataset160npai = dataset160[dataset160['type_erreur'] == 'NPAI']

# Analyse des retours
dataset161delivres = dataset161[dataset161['type_retour'] == 'Délivrés']
dataset161cliques = dataset161[dataset161['type_retour'] == 'Cliqués']
dataset161ouverts = dataset161[dataset161['type_retour'] == 'Ouverts']
dataset161rejetes = dataset161[dataset161['type_retour'] == 'Rejetés']

# Nombre de mandats actifs (qui l'ont été à un moment donné du mois)
dataset164simple = dataset164[dataset164['mandat_type'] == 'Simple (O)']
dataset164demande = dataset164[dataset164['mandat_type'] == 'Demande']
dataset164exclusif = dataset164[dataset164['mandat_type'] == 'Exclusif (O)']
dataset164coexclusif = dataset164[dataset164['mandat_type'] == 'Co exclusif (O)']
dataset164triexclusif = dataset164[dataset164['mandat_type'] == 'Tri exclusif (O)']
dataset164preferentiel = dataset164[dataset164['mandat_type'] == 'Préférentiel (O)']
dataset164delegation = dataset164[dataset164['mandat_type'] == 'Délégation (O)']
dataset164quadriexclusif = dataset164[dataset164['mandat_type'] == 'Quadri exclusif (O)']
# Les autres types de mandats représentent bien trop peu de mandats pour faire des prévisions utiles

In [87]:
# En commentaire le nom du graphique correspondant à ce jeu de données pour les prévisions et son identifiant
# On définira ça au propre à la fin

datasets_noms = [
    'dataset146visite', # 168 Prévisions visite (intervalle de confiance à 80%)
    'dataset146proposition', # 169 Prévisions proposition PAS DE DONNEES
    'dataset146rdv', # 170 Prévisions rendez-vous
    'dataset147', # 171 Prévisions nombre de transactions
    'dataset148', # 172 Prévisions demande en surface
    'dataset149', # 173 Prévisions demande en nombre
    'dataset150', # 174 Prévisions offre en surface
    'dataset151', # 175 Prévisions offre en nombre
    'dataset152', # 176 Prévisions offres entrées
    'dataset153', # 177 Prévisions demandes entrées
    'dataset154', # 178 Prévisions plaintes et désinscriptions PAS DE DONNEES
    'dataset155', # 179 Prévisions total emails envoyés PAS DE DONNEES
    'dataset156action1', # 180 Prévisions action 1
    'dataset156action2', # 181 Prévisions action 2
    'dataset156action3', # 182 Prévisions action 3
    'dataset156action4', # 183 Prévisions action 4
    'dataset156action5', # 184 Prévisions action 5
    'dataset157', # 185 Prévisions nombre de mails par opération PAS DE DONNEES
    'dataset158offres', # 186 Prévisions activité globale offres
    'dataset158demandes', # 187 Prévisions activité globale demandes
    'dataset158mandats', # 188 Prévisions activité globale mandats
    'dataset159offres', # 189 Prévisions activité de marché offres
    'dataset159demandes', # 190 Prévisions activité de marché demandes
    'dataset159mandats', # 191 Prévisions activité de marché mandats
    'dataset160rejete', # 192 Prévisions erreurs : rejeté PAS DE DONNEES
    'dataset160npai', # 193 Prévisions erreurs : NPAI PAS DE DONNEES
    'dataset161delivres', # 194 Prévisions retours : délivrés PAS DE DONNEES
    'dataset161cliques', # 195 Prévisions retours : cliqués PAS DE DONNEES
    'dataset161ouverts', # 196 Prévisions retours : ouverts PAS DE DONNEES
    'dataset161rejetes', # 197 Prévisions retours : rejetés PAS DE DONNEES
    'dataset164simple', # 198 Prévisions mandats simples
    'dataset164demande', # 199 Prévisions mandats demandes
    'dataset164exclusif', # 200 Prévisions mandats exclusifs
    'dataset164coexclusif', # 201 Prévisions mandats coexclusifs
    'dataset164triexclusif', # 202 Prévisions mandats triexclusifs
    'dataset164preferentiel', # 203 Prévisions mandats préférentiels
    'dataset164delegation', # 204 Prévisions mandats délégation
    'dataset164quadriexclusif' # 205 Prévisions mandats quadriexclusifs
]

datasets = [
    dataset146visite,
    dataset146proposition,
    dataset146rdv,
    dataset147,
    dataset148,
    dataset149,
    dataset150,
    dataset151,
    dataset152,
    dataset153,
    dataset154, 
    dataset155,
    dataset156action1,
    dataset156action2,
    dataset156action3,
    dataset156action4,
    dataset156action5,
    dataset157,
    dataset158offres,
    dataset158demandes,
    dataset158mandats,
    dataset159offres,
    dataset159demandes,
    dataset159mandats,
    dataset160rejete,
    dataset160npai,
    dataset161delivres,
    dataset161cliques,
    dataset161ouverts,
    dataset161rejetes,
    dataset164simple,
    dataset164demande,
    dataset164exclusif,
    dataset164coexclusif,
    dataset164triexclusif,
    dataset164preferentiel,
    dataset164delegation,
    dataset164quadriexclusif
]

# On va supprimer de tous les datasets toutes ces colonnes, qui ne sont ni des colonnes date, ni valeur, ni groupe, bref,
# on ne va garder que les données qui nous intéresse, le minimum nécessaire
colonnes_a_abandonner = ['agence_nom', 
                         'agence_id', 
                         'negociateur_nom', 
                         'negociateur_id', 
                         'type_action',
                         'pct_plaintes',
                         'pct_desinscriptions',
                         'nb_plaintes',
                         'nb_desinscriptions',
                         'type_erreur',
                         'type_retour',
                         'mandat_type',
                         'nb_operations',
                         'nb_total_emails'
                         ]

In [88]:
# Suppression des colonnes inutiles pour la prévision
for i, nom_var in enumerate(datasets_noms):
    df = globals()[nom_var] # Permet de modifier la variable globale, et pas seulement le dataset tel qu'il est dans datasets_noms
    df_clean = df.drop(columns=colonnes_a_abandonner, errors='ignore')
    globals()[nom_var] = df_clean # Permet de modifier la variable globale, et pas seulement le dataset tel qu'il est dans la liste datasets

datasets = [globals()[nom] for nom in datasets_noms] # On modifie aussi le contenu du dictionnaire

# Vérification que tous les datasets sont bien sous la forme suivante : [date, groupe_id, série_temp]
for i in range(len(datasets)) : 
    if len(datasets[i].columns) != 3 :
        print(f'AAAAAHHHHH le dataset {datasets_noms[i]} contient {len(datasets[i].columns)} colonnes :')
        print(datasets[i].columns)
        # Si rien ne s'affiche en sortie de cette cellule, c'est bon signe, ça veut dire que toutes les séries temp
        # sont sous la bonne forme

In [89]:
# On lance un dictionnaire pour stocker les nouveaux datasets créés
# La syntaxe sera la suivante : dataset146visite_grp38

datasets_groupes = {}

for i, df in enumerate(datasets):
    dataset_nom = datasets_noms[i]

    # Vérification la présence de la colonne 'groupe_id', normalement tout est bon ici
    if 'groupe_id' not in df.columns:
        print(f"❌ 'groupe_id' manquant dans {dataset_nom}") # En vrai c'est cool les pictogrammes c'est tape-à-l'oeil
        continue

    # Pour chaque groupe, on crée un nouveau dataset (sous réserve d'existence des données)
    for groupe in groupes:
        df_grp = df[df['groupe_id'] == groupe].copy() # On ne garde que les données qui concernent ce groupe en particulier
        if not df_grp.empty:
            nom_dataset = f"{dataset_nom}_grp{groupe}" # On construit le nom du nouveau dataset groupé
            globals()[nom_dataset] = df_grp  # Création d'une variable globale avec ce nom
            datasets_groupes[nom_dataset] = df_grp  # On met aussi dans le dico
            print(f"✅ Créé : {nom_dataset} ({len(df_grp)} lignes)") # Pictooo
        else:
            print(f"⚠️ {dataset_nom} : aucun enregistrement pour groupe {groupe}") # PICTOOOOO

print(f'Il y a pour l\'instant {len(datasets_groupes)} séries temporelles potentiellement analysables, ce nombre va diminuer quand on regardera le nombre de données disponibles')

⚠️ dataset146visite : aucun enregistrement pour groupe 0
✅ Créé : dataset146visite_grp5 (108 lignes)
⚠️ dataset146visite : aucun enregistrement pour groupe 7
✅ Créé : dataset146visite_grp13 (4265 lignes)
✅ Créé : dataset146visite_grp14 (1214 lignes)
✅ Créé : dataset146visite_grp18 (12 lignes)
✅ Créé : dataset146visite_grp29 (1176 lignes)
✅ Créé : dataset146visite_grp35 (2119 lignes)
✅ Créé : dataset146visite_grp38 (29248 lignes)
✅ Créé : dataset146visite_grp39 (412 lignes)
⚠️ dataset146visite : aucun enregistrement pour groupe 40
✅ Créé : dataset146visite_grp43 (1402 lignes)
✅ Créé : dataset146visite_grp44 (1747 lignes)
✅ Créé : dataset146visite_grp45 (282 lignes)
⚠️ dataset146visite : aucun enregistrement pour groupe 46
✅ Créé : dataset146visite_grp47 (1218 lignes)
✅ Créé : dataset146visite_grp48 (3 lignes)
✅ Créé : dataset146visite_grp50 (4059 lignes)
✅ Créé : dataset146visite_grp53 (393 lignes)
✅ Créé : dataset146visite_grp54 (1065 lignes)
✅ Créé : dataset146visite_grp55 (11 lignes)

In [90]:
# On met les colonnes de dates au format datetime, pour que ce soit analysable par Prophet

for nom_df, df in datasets_groupes.items() :
    colonnes_date_type = df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns.tolist()
    colonnes_date_nom = [col for col in df.columns if 'date' in col.lower() or 'jour' in col.lower() or 'mois' in col.lower()]
    
    colonnes_date = [col for col in df.columns if col in colonnes_date_type or col in colonnes_date_nom]

    if not colonnes_date :
        print(f"❌ Aucune colonne date trouvée dans {nom_df}")
        continue

    colonne_date = colonnes_date[0]  # Choix de la première colonne détectée
    df[colonne_date] = pd.to_datetime(df[colonne_date], errors='coerce')

    if colonne_date != 'date' :
        df.rename(columns={colonne_date: 'date'}, inplace=True)

    datasets_groupes[nom_df] = df
    globals()[nom_df] = df


# On teste que les colonnes soient bien toutes au format datetime
for nom, df in datasets_groupes.items() :
    try:
        if not pd.api.types.is_datetime64_any_dtype(df['date']) :
            print(f"{nom} : ⚠️ colonne 'date' n'est pas de type datetime (type actuel : {df['date'].dtype})")
    except KeyError :
        print(f"{nom} : Pas de colonne date")

# Encore une fois, si rien ne s'affiche après cette cellule c'est bon signe (normalement on a fait tous les traitements
# nécessaires pour que rien ne s'affiche)

In [91]:
# On agrège par mois

for nom, df in datasets_groupes.items() : # On parcourt le dictionnaire
    if 'date' not in df.columns : # n-ème vérif
        print('Pas de colonne date dans {nom}')
        continue
    df['date'] = df['date'].dt.to_period('M').dt.to_timestamp()
    colonnes_valeurs = [col for col in df.columns if col not in ['date', 'groupe_id']]
    if not colonnes_valeurs:
        print(f"{nom} n'a pas de données autres que groupe_id et date...")
    df_grouped = df.groupby(['date', 'groupe_id'])[colonnes_valeurs].sum().reset_index()
    datasets_groupes[nom] = df_grouped
    globals()[nom] = df_grouped

    print(f"✅ {nom} : groupé par mois ({len(df_grouped)} lignes)")

✅ dataset146visite_grp5 : groupé par mois (46 lignes)
✅ dataset146visite_grp13 : groupé par mois (171 lignes)
✅ dataset146visite_grp14 : groupé par mois (162 lignes)
✅ dataset146visite_grp18 : groupé par mois (8 lignes)
✅ dataset146visite_grp29 : groupé par mois (171 lignes)
✅ dataset146visite_grp35 : groupé par mois (175 lignes)
✅ dataset146visite_grp38 : groupé par mois (196 lignes)
✅ dataset146visite_grp39 : groupé par mois (70 lignes)
✅ dataset146visite_grp43 : groupé par mois (110 lignes)
✅ dataset146visite_grp44 : groupé par mois (134 lignes)
✅ dataset146visite_grp45 : groupé par mois (76 lignes)
✅ dataset146visite_grp47 : groupé par mois (146 lignes)
✅ dataset146visite_grp48 : groupé par mois (1 lignes)
✅ dataset146visite_grp50 : groupé par mois (149 lignes)
✅ dataset146visite_grp53 : groupé par mois (88 lignes)
✅ dataset146visite_grp54 : groupé par mois (105 lignes)
✅ dataset146visite_grp55 : groupé par mois (4 lignes)
✅ dataset146visite_grp56 : groupé par mois (131 lignes)
✅ d

In [92]:
# Pour plus de simplicité par la suite, on renomme en 'valeur' la colonne de valeurs de chaque série temp

for nom, df in datasets_groupes.items():
    colonnes_a_renommer = [col for col in df.columns if col not in ['date', 'groupe_id']]
    if len(colonnes_a_renommer) != 1:
        print(f"{nom} : {len(colonnes_a_renommer)} IL Y A PLUS D'UNE COLONNE DE VALEURS !!!") # n-ème vérif
        continue  # On ne renomme que s'il y a une seule colonne à renommer
    df = df.rename(columns={colonnes_a_renommer[0]: 'valeur'})
    datasets_groupes[nom] = df
    globals()[nom] = df

# Si rien ne s'affiche après cette cellule c'est bon signe

print(dataset146visite_grp38.columns)

Index(['date', 'groupe_id', 'valeur'], dtype='object')


In [ ]:
# On supprime les données aberrantes

for nom_groupe, df in datasets_groupes.items():
    # Suppression des NaNs si jamais
    df = df.dropna(subset=['valeur'])

    # Détection des outliers via IQR (interquantile range, bref, on regarde à quel point la distribution est dispersée)
    Q1 = df['valeur'].quantile(0.25)
    Q3 = df['valeur'].quantile(0.75)
    IQR = Q3 - Q1

    borne_basse = Q1 - 4 * IQR # 5 c'est énorme mais avec la croissance de certaines séries temp qui partent de bas
    borne_haute = Q3 + 4 * IQR # et les forts effets de saisonnalité, on risque d'avoir régulièrement des valeurs détectées
                               # comme outliers alors qu'elles n'en sont pas si on n'est pas souple

    # Suppression des outliers
    df_sans_outliers = df[(df['valeur'] >= borne_basse) & (df['valeur'] <= borne_haute)]

    datasets_groupes[nom_groupe] = df_sans_outliers

    if nom_groupe in globals():
        globals()[nom_groupe] = df_sans_outliers

    if len(df) - len(df_sans_outliers) != 0 :
        print(f"{nom_groupe} : {len(df) - len(df_sans_outliers)} outliers supprimés sur {len(df)} lignes.")


dataset146visite_grp76 : 1 outliers supprimés sur 107 lignes.
dataset146visite_grp93 : 1 outliers supprimés sur 5 lignes.
dataset146visite_grp154 : 1 outliers supprimés sur 5 lignes.
dataset146visite_grp156 : 1 outliers supprimés sur 5 lignes.
dataset146proposition_grp29 : 1 outliers supprimés sur 6 lignes.
dataset146proposition_grp47 : 1 outliers supprimés sur 32 lignes.
dataset146proposition_grp50 : 1 outliers supprimés sur 29 lignes.
dataset146proposition_grp75 : 1 outliers supprimés sur 8 lignes.
dataset146rdv_grp89 : 2 outliers supprimés sur 10 lignes.
dataset147_grp5 : 1 outliers supprimés sur 8 lignes.
dataset147_grp67 : 2 outliers supprimés sur 123 lignes.
dataset147_grp79 : 5 outliers supprimés sur 61 lignes.
dataset147_grp92 : 1 outliers supprimés sur 76 lignes.
dataset147_grp93 : 2 outliers supprimés sur 9 lignes.
dataset147_grp96 : 1 outliers supprimés sur 16 lignes.
dataset147_grp107 : 2 outliers supprimés sur 182 lignes.
dataset147_grp109 : 2 outliers supprimés sur 68 lig

In [94]:
# On vire les datasets qui contiennent trop peu de données, les prévisions ne seront pas fiables

a_supprimer = []

print(f'Avant les traitements qui viennent, pour rappel, on a {len(datasets_groupes)} séries temporelles à analyser')

for nom, df in datasets_groupes.items() : 
    if df.shape[0] < 36 : # 0 et pas 1, sinon on supprime toutes les données !!!
        a_supprimer.append(nom) # On ajoute dans la liste des datasets non exploitables les datasets qui ont moins de 36 points

Avant les traitements qui viennent, pour rappel, on a 1851 séries temporelles à analyser


In [ ]:
# On vire aussi les datasets qui ont trop de trous

mois_courant = pd.Timestamp(datetime.today().replace(day=1))
seuil_trous = 0.07 # Plus de 7% de trous c'est chaud

for nom, df in datasets_groupes.items() :
    # Génère une série complète de dates mensuelles entre les bornes
    date_range_complet = pd.date_range(start=df['date'].min(), end=mois_courant, freq='MS')

    # Dates présentes dans le DataFrame
    dates_presentes = pd.to_datetime(df['date'].unique())

    nb_total = len(date_range_complet)
    nb_manquantes = len(set(date_range_complet) - set(dates_presentes))
    proportion_trous = nb_manquantes / nb_total
    if proportion_trous > seuil_trous :
        a_supprimer.append(nom)

for nom in set(a_supprimer) : 
    del datasets_groupes[nom]
    del globals()[nom] # On supprime aussi la variable globale pour libérer un peu de mémoire
    print(f"SUPPRIME : {nom}")

print(f'{len(a_supprimer)} datasets supprimés : il ne reste plus que {len(datasets_groupes)} datasets')
# Là normalement avec ça et les datasets trop courts ca aura pas mal baissé

SUPPRIME : dataset156action5_grp55
SUPPRIME : dataset159offres_grp164
SUPPRIME : dataset152_grp111
SUPPRIME : dataset146rdv_grp76
SUPPRIME : dataset159offres_grp124
SUPPRIME : dataset148_grp121
SUPPRIME : dataset148_grp144
SUPPRIME : dataset148_grp120
SUPPRIME : dataset156action4_grp171
SUPPRIME : dataset150_grp45
SUPPRIME : dataset159offres_grp57
SUPPRIME : dataset152_grp135
SUPPRIME : dataset164demande_grp39
SUPPRIME : dataset164delegation_grp89
SUPPRIME : dataset156action5_grp157
SUPPRIME : dataset159demandes_grp152
SUPPRIME : dataset156action4_grp63
SUPPRIME : dataset164demande_grp100
SUPPRIME : dataset156action1_grp108
SUPPRIME : dataset164triexclusif_grp164
SUPPRIME : dataset146visite_grp101
SUPPRIME : dataset156action3_grp144
SUPPRIME : dataset159demandes_grp121
SUPPRIME : dataset150_grp146
SUPPRIME : dataset156action3_grp161
SUPPRIME : dataset153_grp39
SUPPRIME : dataset164delegation_grp72
SUPPRIME : dataset148_grp63
SUPPRIME : dataset156action3_grp57
SUPPRIME : dataset152_grp1

In [ ]:
# Maintenant on supprime aussi les datasets qui ont trop de trous d'affilée, ca ne devrait pas en faire bcp

def mois_vides_consecutifs(dates):
    periods = dates.dt.to_period('M').sort_values().unique()
    full_range = pd.period_range(start=periods.min(), end=mois_courant.to_period('M'), freq='M')
    mois_sans_donnees = [p for p in full_range if p not in periods]

    count, seq_max = 0, 0
    for i in range(1, len(mois_sans_donnees)):
        if mois_sans_donnees[i] - mois_sans_donnees[i - 1] == 1:
            count += 1
            seq_max = max(seq_max, count)
        else:
            count = 0
    return seq_max + 1 >= 3  # +1 car une séquence de 2 gaps = 3 mois consécutifs

datasets_trop_de_trous = [
    nom for nom, df in datasets_groupes.items()
    if 'date' in df.columns and mois_vides_consecutifs(df['date'])
]

for nom in datasets_trop_de_trous : 
    del datasets_groupes[nom]
    del globals()[nom]
    print(f"SUPPRIME : {nom}")

print(f'{len(datasets_trop_de_trous)} datasets supprimés : il ne reste plus que {len(datasets_groupes)} datasets')

0 datasets supprimés : il ne reste plus que 641 datasets


In [ ]:
# ATTENTION !! Nombre de mois manquants à la fin de la série à modifier en fonction du temps depuis laquel la base dev n'a 
# pas été actualisée... Si ca fait longtemps on risque de supprimer tous les datasets en étant trop strict sur le nombre 
# maximal de mois manquants à la fin

# Les prévisions Propeht dépendant plus fortement des derniers mois, ça pose problème si il manque les deux derniers mois
# ou plus. On supprime donc les datasets qui n'ont pas de données récentes.

# Liste des clés à supprimer
cles_a_supprimer = []

# Détermination des 4 derniers mois
mois_actuel = pd.to_datetime(datetime.today().strftime('%Y-%m-01'))
mois_precedent = mois_actuel - pd.DateOffset(months=1)
mois_moins_2 = mois_actuel - pd.DateOffset(months=2)
mois_moins_3 = mois_actuel - pd.DateOffset(months=3)

# On veut garder seulement les datasets ayant AU MOINS UNE donnée dans ces 4 mois
mois_recents = [
    mois_moins_3.to_period('M'),
    mois_moins_2.to_period('M'),
    mois_precedent.to_period('M'),
    mois_actuel.to_period('M')
]

for nom, df in datasets_groupes.items():
    dates_disponibles = pd.to_datetime(df['date']).dt.to_period('M')

    # On met dans la liste des datasets à supprimer si jamais on n'a pas assez de données dans les derniers mois
    if all(mois not in dates_disponibles.values for mois in mois_recents):
        print(f"{nom} : les 4 derniers mois sont absents")
        cles_a_supprimer.append(nom)

# Suppression des datasets
for cle in cles_a_supprimer:
    datasets_groupes.pop(cle, None)
    if cle in globals():
        del globals()[cle]

print(f"{len(cles_a_supprimer)} datasets supprimés.")

dataset147_grp39 : les 4 derniers mois sont absents
dataset150_grp72 : les 4 derniers mois sont absents
dataset156action3_grp39 : les 4 derniers mois sont absents
dataset164delegation_grp92 : les 4 derniers mois sont absents
4 datasets supprimés.


In [98]:
# On supprime aussi les datasets qui ont trop de fois la même valeur, sinon prophet ne saura pas fit de modele

seuil_valeurs_identiques = 0.90
cles_a_supprimer = []

for nom, df in datasets_groupes.items():
    valeurs_uniques = df['valeur'].value_counts(normalize=True)
    if any(valeurs_uniques > seuil_valeurs_identiques):
        # print(f"Suppression de {nom} : une valeur apparaît dans plus de 1/3 des cas")
        cles_a_supprimer.append(nom)

print(f'{len(cles_a_supprimer)} datasets ont été supprimés car trop de valeurs identiques')

for cle in cles_a_supprimer:
    datasets_groupes.pop(cle, None)
    if cle in globals():
        del globals()[cle]


13 datasets ont été supprimés car trop de valeurs identiques


In [ ]:
# Petite vérif avant d'interpoler les valeurs manquantes

for nom, df in datasets_groupes.items() :
    if len(df) < 36 :
        print(f"❗ {nom} contient seulement {len(df)} lignes avant interpolation")

In [100]:
# Interpolation des valeurs manquantes

for nom, df in datasets_groupes.items():
    mois_complets = pd.date_range(start=df['date'].min(), end=df['date'].max(), freq='MS')
    groupe = df['groupe_id'].iloc[0]
    df_toutes_dates = pd.DataFrame({'date': mois_complets})
    df_toutes_dates['groupe_id'] = groupe
    df_fusionne = df_toutes_dates.merge(df, on=['date', 'groupe_id'], how='left')
    df_fusionne['valeur'] = df_fusionne['valeur'].interpolate(method='linear')
    datasets_groupes[nom] = df_fusionne
    globals()[nom] = df_fusionne
    print(f"✅ {nom} complété ({len(df_fusionne)} mois)")

✅ dataset146visite_grp35 complété (176 mois)
✅ dataset146visite_grp38 complété (203 mois)
✅ dataset146visite_grp47 complété (147 mois)
✅ dataset146visite_grp50 complété (151 mois)
✅ dataset146visite_grp56 complété (134 mois)
✅ dataset146visite_grp67 complété (120 mois)
✅ dataset146visite_grp68 complété (120 mois)
✅ dataset146visite_grp75 complété (112 mois)
✅ dataset146visite_grp76 complété (108 mois)
✅ dataset146visite_grp79 complété (109 mois)
✅ dataset146visite_grp83 complété (103 mois)
✅ dataset146visite_grp85 complété (102 mois)
✅ dataset146visite_grp86 complété (99 mois)
✅ dataset146visite_grp118 complété (57 mois)
✅ dataset146rdv_grp13 complété (202 mois)
✅ dataset146rdv_grp35 complété (175 mois)
✅ dataset146rdv_grp83 complété (55 mois)
✅ dataset146rdv_grp118 complété (55 mois)
✅ dataset146rdv_grp153 complété (172 mois)
✅ dataset147_grp13 complété (202 mois)
✅ dataset147_grp14 complété (202 mois)
✅ dataset147_grp35 complété (202 mois)
✅ dataset147_grp38 complété (204 mois)
✅ dat

In [101]:
# Pour ne faire tourner qu'à partir d'un certain dataset (remplacer alors dans la cellule suivante datasets_groupes par nouveau_dico)

start_key = 'dataset164triexclusif_grp76'
trouver = False
nouveau_dico = {}

for key in datasets_groupes.keys() :
    if key == start_key:
        trouver = True
    if trouver:
        nouveau_dico[key] = datasets_groupes[key]

print(nouveau_dico.keys())
print(len(nouveau_dico))

dict_keys(['dataset164triexclusif_grp76', 'dataset164triexclusif_grp79', 'dataset164triexclusif_grp86', 'dataset164triexclusif_grp89', 'dataset164triexclusif_grp108', 'dataset164triexclusif_grp124', 'dataset164triexclusif_grp133', 'dataset164triexclusif_grp153', 'dataset164preferentiel_grp35', 'dataset164preferentiel_grp38', 'dataset164preferentiel_grp56', 'dataset164preferentiel_grp64', 'dataset164preferentiel_grp68', 'dataset164preferentiel_grp79', 'dataset164preferentiel_grp86', 'dataset164delegation_grp13', 'dataset164delegation_grp38', 'dataset164delegation_grp47', 'dataset164delegation_grp50', 'dataset164delegation_grp56', 'dataset164delegation_grp86', 'dataset164delegation_grp114', 'dataset164delegation_grp153', 'dataset164quadriexclusif_grp13', 'dataset164quadriexclusif_grp38', 'dataset164quadriexclusif_grp39', 'dataset164quadriexclusif_grp50', 'dataset164quadriexclusif_grp56', 'dataset164quadriexclusif_grp114', 'dataset164quadriexclusif_grp133'])
30


In [102]:
keys = list(datasets_groupes.keys())
print(f"Index de départ : {keys.index(start_key)}")
print(f"Clés à partir de start_key : {keys[keys.index(start_key):]}")
print(f"Nombre attendu : {len(keys[keys.index(start_key):])}")

Index de départ : 594
Clés à partir de start_key : ['dataset164triexclusif_grp76', 'dataset164triexclusif_grp79', 'dataset164triexclusif_grp86', 'dataset164triexclusif_grp89', 'dataset164triexclusif_grp108', 'dataset164triexclusif_grp124', 'dataset164triexclusif_grp133', 'dataset164triexclusif_grp153', 'dataset164preferentiel_grp35', 'dataset164preferentiel_grp38', 'dataset164preferentiel_grp56', 'dataset164preferentiel_grp64', 'dataset164preferentiel_grp68', 'dataset164preferentiel_grp79', 'dataset164preferentiel_grp86', 'dataset164delegation_grp13', 'dataset164delegation_grp38', 'dataset164delegation_grp47', 'dataset164delegation_grp50', 'dataset164delegation_grp56', 'dataset164delegation_grp86', 'dataset164delegation_grp114', 'dataset164delegation_grp153', 'dataset164quadriexclusif_grp13', 'dataset164quadriexclusif_grp38', 'dataset164quadriexclusif_grp39', 'dataset164quadriexclusif_grp50', 'dataset164quadriexclusif_grp56', 'dataset164quadriexclusif_grp114', 'dataset164quadriexclusif

In [103]:
moins_de_100 = sum(len(df) < 100 for df in datasets_groupes.values())
total_series = len(datasets_groupes)

print(f"La proportion de séries qui ont moins de 100 points est de {(moins_de_100 / total_series) * 100:.2f}%")

La proportion de séries qui ont moins de 100 points est de 24.20%


In [104]:
print(nouveau_dico.keys()) # Prour regarder le nom de la derniere serie et verifier que tout a bien ete fait

dict_keys(['dataset164triexclusif_grp76', 'dataset164triexclusif_grp79', 'dataset164triexclusif_grp86', 'dataset164triexclusif_grp89', 'dataset164triexclusif_grp108', 'dataset164triexclusif_grp124', 'dataset164triexclusif_grp133', 'dataset164triexclusif_grp153', 'dataset164preferentiel_grp35', 'dataset164preferentiel_grp38', 'dataset164preferentiel_grp56', 'dataset164preferentiel_grp64', 'dataset164preferentiel_grp68', 'dataset164preferentiel_grp79', 'dataset164preferentiel_grp86', 'dataset164delegation_grp13', 'dataset164delegation_grp38', 'dataset164delegation_grp47', 'dataset164delegation_grp50', 'dataset164delegation_grp56', 'dataset164delegation_grp86', 'dataset164delegation_grp114', 'dataset164delegation_grp153', 'dataset164quadriexclusif_grp13', 'dataset164quadriexclusif_grp38', 'dataset164quadriexclusif_grp39', 'dataset164quadriexclusif_grp50', 'dataset164quadriexclusif_grp56', 'dataset164quadriexclusif_grp114', 'dataset164quadriexclusif_grp133'])


In [105]:
liste_noms = list(datasets_groupes.keys())
for nom in liste_noms :
    if nom in nouveau_dico.keys() :
        nouveau_dico[nom] = nouveau_dico[nom].rename(columns={'date' : 'ds', 'valeur' : 'y'})
    datasets_groupes[nom] = datasets_groupes[nom].rename(columns={'date' : 'ds', 'valeur' : 'y'})
    print(f"{nom} : {datasets_groupes[nom].columns}")

dataset146visite_grp35 : Index(['ds', 'groupe_id', 'y'], dtype='object')
dataset146visite_grp38 : Index(['ds', 'groupe_id', 'y'], dtype='object')
dataset146visite_grp47 : Index(['ds', 'groupe_id', 'y'], dtype='object')
dataset146visite_grp50 : Index(['ds', 'groupe_id', 'y'], dtype='object')
dataset146visite_grp56 : Index(['ds', 'groupe_id', 'y'], dtype='object')
dataset146visite_grp67 : Index(['ds', 'groupe_id', 'y'], dtype='object')
dataset146visite_grp68 : Index(['ds', 'groupe_id', 'y'], dtype='object')
dataset146visite_grp75 : Index(['ds', 'groupe_id', 'y'], dtype='object')
dataset146visite_grp76 : Index(['ds', 'groupe_id', 'y'], dtype='object')
dataset146visite_grp79 : Index(['ds', 'groupe_id', 'y'], dtype='object')
dataset146visite_grp83 : Index(['ds', 'groupe_id', 'y'], dtype='object')
dataset146visite_grp85 : Index(['ds', 'groupe_id', 'y'], dtype='object')
dataset146visite_grp86 : Index(['ds', 'groupe_id', 'y'], dtype='object')
dataset146visite_grp118 : Index(['ds', 'groupe_id',

In [ ]:
# On initialise un dataframe de statistiques descriptives sur les séries avant de faire les prévisions.
# On va essayer de les clusteriser là-dessus, en ajoutant d'autres indicateurs qui leur sont propre (variance, etc.)
# On commence par boucler sur les séries pour leur ajouter leur variance dans le tableau

# En fait c'est de l'extraction de statistiques descriptives sur le jeu de données

from scipy.stats import skew, kurtosis

series_temp_features = []

for nom, df in datasets_groupes.items():
    match = re.search(r'grp(\d+)', nom)
    groupe_id = int(match.group(1)) if match else None
    longueur_serie = df.shape[0]
    
    y = df['y'].dropna().reset_index(drop=True)
    moyenne = y.mean()
    std = y.std()
    std_normalise = std / moyenne if moyenne != 0 else None

    # Découper la série en 2 moitiés
    n = len(y)
    mid = n // 2
    y_first = y.iloc[:mid]
    y_second = y.iloc[mid:]
    
    moyenne1 = y_first.mean()
    moyenne2 = y_second.mean()
    
    std1 = y_first.std() / moyenne1 if moyenne1 != 0 else None
    std2 = y_second.std() / moyenne2 if moyenne2 != 0 else None

    # Moyennes locales pour le ratio début / fin
    moyenne_debut = y.iloc[:6].mean() if len(y) >= 6 else None
    moyenne_fin = y.iloc[-9:-3].mean() if len(y) >= 9 else None
    ratio_debut_fin = moyenne_debut / moyenne_fin if moyenne_fin and moyenne_fin != 0 else None

    # Assymétrie et aplatissement (skewness et kurtosis)
    skewness_val = skew(y) if len(y) >= 3 else None
    kurtosis_val = kurtosis(y) if len(y) >= 4 else None  # sinon division par zéro potentielle

    # Test de stationnarité ADF
    try :
        adf_result = adfuller(y.dropna())
        adf_pvalue = adf_result[1]  # p-value
    except Exception:
        adf_pvalue = None

    # On calcule les composantes de Fourier
    fft_vals = fft(y - y.mean())  # centrer la série avant fft
    fft_freqs = np.fft.fftfreq(len(y))
    positive_freq_idxs = np.where(fft_freqs > 0)

    if len(positive_freq_idxs[0]) > 0:
        dominant_idx = positive_freq_idxs[0][np.argmax(np.abs(fft_vals[positive_freq_idxs]))]
        dominant_freq = fft_freqs[dominant_idx]
        dominant_amp = np.abs(fft_vals[dominant_idx])
        dominant_phase = np.angle(fft_vals[dominant_idx])
    else:
        dominant_freq = None
        dominant_amp = None
        dominant_phase = None

    # On crée un histogramme normalisé des valeurs
    hist, bin_edges = np.histogram(y, bins=10, density=True)
    hist = hist + 1e-12  # éviter log(0)
    shannon_entropy = entropy(hist)

    series_temp_features.append({
        'nom_serie': nom,
        'groupe_id': groupe_id,
        'longueur_serie': longueur_serie,
        'mean_abs_diff' : np.mean(np.abs(np.diff(y))),
        'std_normalise': std_normalise,
        'std_normalise_first_half': std1,
        'std_normalise_second_half': std2,
        'std_normalise_ecart': abs(std1 - std2) if std1 is not None and std2 is not None else None,
        'moyenne1': moyenne1,
        'moyenne2': moyenne2,
        'ecart_moyennes_normalise': abs(moyenne1 - moyenne2)/moyenne if moyenne1 is not None and moyenne2 is not None and moyenne != 0 else None,
        'skewness': skewness_val,
        'kurtosis': kurtosis_val,
        'moyenne_debut': moyenne_debut,
        'moyenne_fin': moyenne_fin,
        'ratio_debut_fin': ratio_debut_fin,
        'ecart_interquantile' : y.quantile(0.75) - y.quantile(0.25),
        'autocorr' : y.autocorr(lag=1),
        'autocorrelation_saisonniere' : y.autocorr(lag=12) if len(y) >= 24 else None,
        'turning_points' : np.sum(np.diff(np.sign(np.diff(y))) != 0)/len(y),
        'mediane' : y.median(),
        'tendance' : linregress(range(len(y)), y).slope,
        'adf_pvalue': adf_pvalue,
        'fft_dominant_freq': dominant_freq,
        'fft_dominant_amp': dominant_amp,
        'fft_dominant_phase': dominant_phase,
        'shannon_entropy': shannon_entropy
    })

# Convertir en DataFrame
series_temp_features = pd.DataFrame(series_temp_features)

print(series_temp_features)

                           nom_serie  groupe_id  longueur_serie  \
0             dataset146visite_grp35         35             176   
1             dataset146visite_grp38         38             203   
2             dataset146visite_grp47         47             147   
3             dataset146visite_grp50         50             151   
4             dataset146visite_grp56         56             134   
..                               ...        ...             ...   
619   dataset164quadriexclusif_grp39         39             103   
620   dataset164quadriexclusif_grp50         50             151   
621   dataset164quadriexclusif_grp56         56             204   
622  dataset164quadriexclusif_grp114        114              46   
623  dataset164quadriexclusif_grp133        133              58   

     mean_abs_diff  std_normalise  std_normalise_first_half  \
0         9.908571       0.546602                  0.643550   
1        82.034653       0.661033                  1.052619   
2     

In [107]:
series_temp_features_notebook_balayage = pd.read_excel("series_temp_features.xlsx")

In [108]:
# Récupération des noms de séries déjà présents dans le DataFrame
series_existantes = set(series_temp_features_notebook_balayage['nom_serie'])

# Création du dictionnaire des séries additionnelles
datasets_groupes_additionnels = {
    nom_serie: df
    for nom_serie, df in datasets_groupes.items()
    if nom_serie not in series_existantes
}

In [109]:
# Première méthode : on fait une boucle qui permettra d'analyser une série temporelle du début à la fin automatiquement
# Compter deux heures d'exécution

print(f'{len(datasets_groupes_additionnels)} datasets encore en lice')

# On va stocker là-dedans les prévisions : valeur prédite, borne inférieure de l'intervalle de confiance (95%), borne sup
# de l'intervalle de confiance, l'identifiant du groupe auquel se réfère la prédiction, et enfin la date pour laquelle 
# ces valeurs sont valables
resultats_previsions = []

i = 0 # On initialise un compteur du nombre de séries temporelles qui vont utiliser un mcmc sampling
j = 0 # On initialise un compteur de la somme de toutes les rmsse de toutes les séries temporelles
l = 0 # On initialise un compteur du nombre de séries qui ont pu être traitées

dico_des_rmsse = {cle : None for cle in datasets_groupes_additionnels.keys()}

prior_scale_possibles = [0.01, 0.03, 0.05] # Pour contrôler la vitesse d'adaptation du modèle aux changements de tendance
saisonnalite_oui_non_possibles = [True, False] # Soit on dit qu'il y a de la saisonnalité et on la recherche, soit on dit qu'il n'y en a pas et donc on ne la recherche pas
modes_possibles = ['multiplicative', 'additive'] # multiplicative : la saisonnalité sera modélisée comme la valeur prédite par la tendance fois un certain coeff, et pour additive, ce sera un + qqch

niveau_surajustement = 2

dico_des_methodes = {cle : None for cle in datasets_groupes_additionnels.keys()}

taille_minimale = 60
max_essais = 3

with warnings.catch_warnings() :
    warnings.simplefilter("ignore")

    for nom, df_prophet in datasets_groupes_additionnels.items() :
        # Extraire le groupe_id depuis le nom si possible
        match = re.search(r'grp(\d+)', nom)
        niveau_raccourcissement = None
        nb_essais = 0

        try : 
            recommencer = True

            while recommencer == True and nb_essais < max_essais :
                nb_essais += 1
                prediction_constante = False
                recommencer = False  # par défaut, on ne recommence pas
                forecast = None  # Réinitialiser à chaque itération
                forecast_12 = None
                rmsse_actualisee = None
                id = None # Pour identifier ce qu'il s'est passé dans la boucle
                surajustement = False
                params_et_resultats = [] # On stockera dedans les caractéristiques de chaque modèle testé

                # On ajuste le paramètre de sensibilité de l'adaptation de la tendance détectée aux chocs brutaux
                for prior_scale_param in prior_scale_possibles :
                    # Normalement si on met True et qu'il s'avere qu'il n'y a pas de saisonnalité, le modèle apprendra
                    # que ca ne sert à rien de la prendre en compte et donc mettra les coeffs des composantes saisonnières
                    # à 0. Mais on risque d'interpréter le bruit comme de la saisonnalité, donc autant tester la rmse 
                    # dans les deux cas

                    for saisonnalite_oui_non in saisonnalite_oui_non_possibles :
                        # Multiplicative : la saisonnalité est proportionnellement forte avec la tendance
                        # Additive : la saisonnalité a toujours la même force. 
                        # On peut être dans les deux cas donc on teste les deux

                        for mode in modes_possibles :
                            model = Prophet(seasonality_mode=mode,
                            weekly_seasonality=False,
                            daily_seasonality=False,
                            yearly_seasonality=saisonnalite_oui_non,
                            changepoint_prior_scale=prior_scale_param,
                            n_changepoints=int(np.floor(df_prophet.shape[0]/3)))

                            try :
                                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                                    model.fit(df_prophet)
                            except Exception as e :
                                print(f"Erreur pendant le fit sur {nom} : {e}")
                                continue  # passe à la série suivante

                            # Vérifications
                            if not hasattr(model, "stan_fit") or model.stan_fit is None :
                                print(f"Erreur : model.stan_fit est None pour {nom}")
                                continue
                            if not hasattr(model.stan_fit, "runset") or model.stan_fit.runset is None :
                                print(f"Erreur : model.stan_fit.runset est None pour {nom}")
                                continue

                            # C'est la validation croisée qui prend beaucoup de temps. 
                            # Pour raccourcir le temps d'exécution : 
                            # initial donne la durée du premier intervalle sur lequel on entraîne le modèle.
                            # Plus il est long, moins on aura de validations croisées à faire
                            # period donne le nombre de jours qu'on va ajouter aux données d'entraînement pour la prochaine validation
                            # Plus on en ajoute, moins on fera de validation (donc plus ce sera rapide)
                            # On ne touche pas à horizon, c'est le nombre de mois qu'on prédit, et comme on est sur des données
                            # mensuelles, on prédit sur les douze mois qui viennent

                            nb_mois = len(df_prophet)
                            jours_total = int(nb_mois * 30.44)
                            param_initial = f'{int(jours_total * 0.75)} days'
                            param_period = f'{int(jours_total * 0.20)} days'
                            param_horizon = f'{int(jours_total * 0.05)} days'  # 5% du total

                            df_cv = cross_validation(
                                model,
                                initial=param_initial,
                                period=param_period,
                                horizon=param_horizon,
                                disable_tqdm=True)

                            rmse_provisoire = performance_metrics(df_cv)['rmse'].mean()
                            params_et_resultats.append([prior_scale_param, saisonnalite_oui_non, mode, rmse_provisoire])

                params_et_resultats = np.array(params_et_resultats, dtype=object)
                colonne_rmse = params_et_resultats[:, -1].astype(float)
                meilleur_indice_rmse = np.argmin(colonne_rmse)
                meilleurs_parametres = params_et_resultats[meilleur_indice_rmse]
                use_mcmc = meilleurs_parametres[1] is True

                model_prophet = Prophet(
                    seasonality_mode=meilleurs_parametres[2],
                    weekly_seasonality=False,
                    daily_seasonality=False,
                    yearly_seasonality=meilleurs_parametres[1],
                    n_changepoints=int(np.floor(df_prophet.shape[0]/3)),
                    changepoint_prior_scale = meilleurs_parametres[0],
                    **({'mcmc_samples': 150} if use_mcmc else {}),
                    uncertainty_samples=1000)

                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()) :
                    if use_mcmc :
                        try :
                            model_prophet.fit(df_prophet, show_progress=False)
                        except Exception :
                            pass  # ignore errors silently, optionally log internally
                    else :
                        try :
                            model_prophet.fit(df_prophet)
                        except Exception :
                            pass
                
                # On est obligé de refaire une validation croisée pcq sinon df_cv donne les resultats de la derniere qui a 
                # été faite, et non ceux de la meilleure

                nb_mois = len(df_prophet)
                jours_total = int(nb_mois * 30.44)
                param_initial = f'{int(jours_total * 0.75)} days'
                param_period = f'{int(jours_total * 0.20)} days'
                param_horizon = f'{int(jours_total * 0.05)} days'  # 5% du total

                df_cv = cross_validation(
                    model,
                    initial=param_initial,
                    period=param_period,
                    horizon=param_horizon,
                    disable_tqdm=True)

                # Calcul de la RMSSE (rmse normalisée pour être comparable sur des séries d'échelle très diverses)
                y_test = df_cv['y'].values
                y_pred = df_cv['yhat'].values
                y_train = df_prophet['y'].values
                denom = np.sqrt(np.sum(np.diff(y_train) ** 2) / (len(y_train) - 1))
                rmsse = meilleurs_parametres[3] / denom

                # On a maintenant le meilleur modèle sur la série brute. 
                # Il y a maintenant 4 cas de figure : 
                # Soit les résultats sont satisfaisants et on les garde
                # Soit les résultats sont mauvais et la série fait plus de 96 points : on rétrécit à 60
                # Soit les résultats sont mauvais et la série fait entre 36 et 96 points : on rétrécit à 36
                # Soit la série fait 36 points et les résultats sont toujours mauvais : on tente de surajuster

                # 1er cas : les résultats sont satisfaisants 
                if rmsse < 1 :
                    rmsse_finale = rmsse

                    # Prédiction Prophet sur 12 mois
                    future = model_prophet.make_future_dataframe(periods=12, freq='MS')
                    forecast = model_prophet.predict(future)

                    forecast[['yhat', 'yhat_lower', 'yhat_upper']] = forecast[['yhat', 'yhat_lower', 'yhat_upper']].clip(lower=0)
                    forecast_12 = forecast.tail(12).copy()
                    forecast_12[['yhat', 'yhat_lower', 'yhat_upper']] = forecast_12[['yhat', 'yhat_lower', 'yhat_upper']].clip(lower=0)
                    forecast_12['dataset_nom'] = nom
                    forecast_12['groupe_id'] = int(match.group(1)) if match else None
                    forecast_12 = forecast_12[['ds', 'yhat', 'yhat_lower', 'yhat_upper', 'dataset_nom', 'groupe_id']]

                # 2e cas : les résultats ne sont pas satisfaisants et la série fait plus de 96 points
                if rmsse > 1 and df_prophet.shape[0] > 96 :
                    recommencer = True
                    df_prophet=df_prophet.tail(60)
                    niveau_raccourcissement = '60'
                    continue # On repart du début de la boucle while avec la série raccourcie

                # 3e cas : les résultats ne sont pas satisfaisants et la série fait entre 36 et 96 points
                if rmsse > 1 and 36 < df_prophet.shape[0] <= 96 :
                    recommencer = True
                    df_prophet = df_prophet.tail(36)
                    niveau_raccourcissement = '36'
                    continue # On repart du début de la boucle while avec une série raccourcie

                # 4e et dernier cas : la série fait déjà 36 points mais les résultats ne sont toujours pas satisfaisants
                if rmsse > 1 and df_prophet.shape[0] == 36 :
                    
                    # On surajuste et on voit ce qui se passe
                    # Gros bloc ici parce qu'on refait tout, mais juste sans passer par le balayage des valeur possibles de changepoints_prior_scale
                    surajustement = True
                    params_et_resultats = []

                    # On reparcourt les différents paramètres de saisonnalité possibles
                    for saisonnalite_oui_non in saisonnalite_oui_non_possibles :

                        for mode in modes_possibles :
                            # Ici on recalcule le modèle avec surajustement 
                            model = Prophet(seasonality_mode=mode,
                                weekly_seasonality=False,
                                daily_seasonality=False,
                                yearly_seasonality=saisonnalite_oui_non,
                                changepoint_prior_scale=niveau_surajustement,
                                n_changepoints=int(np.floor(df_prophet.shape[0]/3)))

                            # On refit le modele
                            try :
                                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()) :
                                    model.fit(df_prophet)
                            except Exception as e :
                                print(f"Erreur pendant le fit sur {nom} : {e}")
                                continue  # passe à la série suivante

                            # Vérifications
                            if not hasattr(model, "stan_fit") or model.stan_fit is None :
                                print(f"Erreur : model.stan_fit est None pour {nom}")
                                continue
                            if not hasattr(model.stan_fit, "runset") or model.stan_fit.runset is None :
                                print(f"Erreur : model.stan_fit.runset est None pour {nom}")
                                continue

                            nb_mois = len(df_prophet)
                            jours_total = int(nb_mois * 30.44)
                            param_initial = f'{int(jours_total * 0.75)} days'
                            param_period = f'{int(jours_total * 0.20)} days'
                            param_horizon = f'{int(jours_total * 0.05)} days'  # 5% du total

                            df_cv = cross_validation(
                                model,
                                initial=param_initial,
                                period=param_period,
                                horizon=param_horizon,
                                disable_tqdm=True)

                            rmse_provisoire = performance_metrics(df_cv)['rmse'].mean()
                            params_et_resultats.append([niveau_surajustement, saisonnalite_oui_non, mode, rmse_provisoire])

                    params_et_resultats = np.array(params_et_resultats, dtype=object)
                    colonne_rmse = params_et_resultats[:, -1].astype(float)
                    meilleur_indice_rmse = np.argmin(colonne_rmse)
                    meilleurs_parametres = params_et_resultats[meilleur_indice_rmse]
                    use_mcmc = meilleurs_parametres[1] is True

                    model_prophet = Prophet(
                        seasonality_mode=meilleurs_parametres[2],
                        weekly_seasonality=False,
                        daily_seasonality=False,
                        yearly_seasonality=meilleurs_parametres[1],
                        n_changepoints=int(np.floor(df_prophet.shape[0]/3)),
                        changepoint_prior_scale = meilleurs_parametres[0],
                        **({'mcmc_samples': 150} if use_mcmc else {}),
                        uncertainty_samples=1000)

                    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()) :
                        if use_mcmc :
                            try :
                                model_prophet.fit(df_prophet, show_progress=False)
                            except Exception :
                                pass  # ignore errors silently, optionally log internally
                        else :
                            try :
                                model_prophet.fit(df_prophet)
                            except Exception :
                                pass
                    
                    # On est obligé de refaire une validation croisée pcq sinon df_cv donne les resultats de la derniere qui a 
                    # été faite, et non ceux de la meilleure

                    nb_mois = len(df_prophet)
                    jours_total = int(nb_mois * 30.44)
                    param_initial = f'{int(jours_total * 0.75)} days'
                    param_period = f'{int(jours_total * 0.20)} days'
                    param_horizon = f'{int(jours_total * 0.05)} days'  # 5% du total

                    df_cv = cross_validation(
                        model,
                        initial=param_initial,
                        period=param_period,
                        horizon=param_horizon,
                        disable_tqdm=True)

                    # Calcul de la RMSSE (rmse normalisée pour être comparable sur des séries d'échelle très diverses)
                    y_test = df_cv['y'].values
                    y_pred = df_cv['yhat'].values
                    y_train = df_prophet['y'].values
                    denom = np.sqrt(np.sum(np.diff(y_train) ** 2) / (len(y_train) - 1))
                    rmsse = meilleurs_parametres[3] / denom

                    # On a maintenant récupéré la rmsse du meilleur modèle surajusté pour la série raccourcie au maximum. 
                    # On a deux possibilités : soit c'est enfin bon (rmsse < 1), soit on abandonne et on fait une prévision constante

                    # 1er cas : ça marche enfin
                    if rmsse < 1 : 
                        surajustement = True
                        rmsse_finale = rmsse
                        niveau_raccourcissement = '36'

                        # Prédiction Prophet sur 12 mois
                        future = model_prophet.make_future_dataframe(periods=12, freq='MS')
                        forecast = model_prophet.predict(future)

                        forecast[['yhat', 'yhat_lower', 'yhat_upper']] = forecast[['yhat', 'yhat_lower', 'yhat_upper']].clip(lower=0)
                        forecast_12 = forecast.tail(12).copy()
                        forecast_12[['yhat', 'yhat_lower', 'yhat_upper']] = forecast_12[['yhat', 'yhat_lower', 'yhat_upper']].clip(lower=0)
                        forecast_12['dataset_nom'] = nom
                        forecast_12['groupe_id'] = int(match.group(1)) if match else None
                        forecast_12 = forecast_12[['ds', 'yhat', 'yhat_lower', 'yhat_upper', 'dataset_nom', 'groupe_id']]

                    # 2e cas : ça ne marche toujours pas : prévision constante
                    if rmsse > 1 :
                        surajustement = False
                        niveau_raccourcssement = '36'
                        use_mcmc = False
                        rmsse_finale = 1

                        valeur_constante = df_prophet['y'].tail(12).mean()
                        future_dates = pd.date_range(start=df_prophet['ds'].max() + pd.DateOffset(months=1), periods=12, freq='MS')

                        forecast_12 = pd.DataFrame({
                            'ds': future_dates,
                            'yhat': [valeur_constante] * 12,
                            'yhat_lower': [valeur_constante] * 12,
                            'yhat_upper': [valeur_constante] * 12,
                            'dataset_nom': nom,
                            'groupe_id': int(match.group(1)) if match else None
                        })

                        future = model_prophet.make_future_dataframe(periods=12, freq='MS')
                        forecast = pd.DataFrame({
                            'ds': future['ds'],
                            'yhat': list(df_prophet['y']) + [valeur_constante] * 12,
                            'yhat_lower': list(df_prophet['y']) + [valeur_constante] * 12,
                            'yhat_upper': list(df_prophet['y']) + [valeur_constante] * 12
                        })

                        prediction_constante = True

            dico_des_rmsse[nom] = rmsse_finale
            
            if nb_essais == 1 and rmsse_finale < 1 :
                id = 'succes_immediat'
            if nb_essais == 2 and df_prophet.shape[0] == 60 and rmsse_finale < 1 :
                id = 'serie_raccourcie_60'
            if nb_essais == 2 and df_prophet.shape[0] == 36 and rmsse_finale < 1 :
                id = 'serie_raccourcie_36'
            if nb_essais == 3 and df_prophet.shape[0] == 36 and rmsse_finale < 1 :
                id = 'serie_raccourcie_36'
            if nb_essais == 3 and df_prophet.shape[0] == 36 and rmsse_finale == 1 :
                id = 'echec'
            if nb_essais == 2 and df_prophet.shape[0] == 36 and rmsse_finale == 1 :
                id = 'echec'

            # Pour éviter de garder des prévisions aberrantes, on ne sait jamais
            if (forecast_12['yhat_upper'] < forecast_12['yhat']).any() :
                forecast_12 = None
                k += 1

            if forecast_12 is not None :
                resultats_previsions.append(forecast_12)
                if use_mcmc :
                    i += 1

            dico_des_methodes[nom] = id

            if id in ['succes_immediat', 'serie_raccourcie_60', 'serie_raccourcie_36'] :
                # fig = model_prophet.plot(forecast)
                # fig.suptitle(nom, fontsize=14)
                # plt.show()
                pass
            if id == 'echec' :
                # fig = plt.figure()
                # plt.plot(df_prophet['ds'], df_prophet['y'], label='Historique')
                # plt.plot(forecast_12['ds'], forecast_12['yhat'], label='Prévision constante')
                # plt.fill_between(forecast_12['ds'], forecast_12['yhat_lower'], forecast_12['yhat_upper'], alpha=0.2)
                # plt.title(f"{nom} (Prédiction constante)")
                # plt.legend()
                # plt.show()
                pass

            # On affiche l'avancement
            print(f"{nom} faite, plus que {len(list(datasets_groupes_additionnels.keys())[list(datasets_groupes_additionnels.keys()).index(nom)+1:])}. RMSSE : {rmsse_finale} ({id}) MCMC : {use_mcmc}, Surajustement : {surajustement}, raccourcissement : {niveau_raccourcissement}")
            if forecast_12 is None :
                print("MAJ : prévisions aberrantes et donc non retenues")
                
            l += 1
            j += rmsse_finale

            # Trouver l'index de la ligne correspondant à la série
            index_ligne = series_temp_features[series_temp_features['nom_serie'] == nom].index

            if not index_ligne.empty :
                index_ligne = index_ligne[0]
                series_temp_features.loc[index_ligne, 'mcmc_utilise'] = use_mcmc
                series_temp_features.loc[index_ligne, 'saisonnalite'] = meilleurs_parametres[1]
                series_temp_features.loc[index_ligne, 'mode_saisonnalite'] = meilleurs_parametres[2] if meilleurs_parametres[1] == True else None
                series_temp_features.loc[index_ligne, 'changepoint_prior_scale'] = meilleurs_parametres[0]
                series_temp_features.loc[index_ligne, 'rmsse'] = rmsse_finale
                series_temp_features.loc[index_ligne, 'id_methode'] = id
                series_temp_features.loc[index_ligne, 'surajustement'] = surajustement
            else:
                print(f"⚠️  La série {nom} n'existe pas dans series_temp_features.")

        except Exception as e :
            print(f'{nom} N\'A PAS PU ÊTRE TRAITEE : {e}')
            recommencer = False

            # Pour éviter un crash plus bas
            use_mcmc = None
            meilleurs_parametres = None
            rmsse_finale = None
            id = None
            surajustement = None

print(f'{i} séries temporelles sur {len(datasets_groupes_additionnels)} ont utilisé un mcmc sampling')

df_previsions_parametres_deja_connus = pd.concat(resultats_previsions, ignore_index=True)

df_previsions_parametres_deja_connus = df_previsions_parametres_deja_connus.rename(columns={
    'dataset_nom': 'graphique_id',
    'groupe_id': 'groupe_id',
    'ds': 'gd_date',
    'yhat': 'gd_y',
    'yhat_lower': 'gd_inf',
    'yhat_upper': 'gd_sup'
})

# ATTENTION !!! LES INTERVALLES DE CONFIANCE OBTENUS [yhat_lower ; yhat_upper] NE SONT PAS FORCEMENT CENTRES EN yhat !!!
# En fait, Prophet se base sur les quantiles des distributions empiriques des trajectoires simulées pour les yhat_lower
# et yhat_upper dans le cas de l'utilisation d'un mcmc sampling et garde la médiane pour yhat. DONC pas de symétrie de
# l'intervalle de confiance !!! Et même pour le maximum a poteriori, on n'a pas de centrage de l'intervalle de confiance

series_temp_features = pd.DataFrame(series_temp_features)

print(dico_des_methodes)
print(f'Somme de toutes les rmsse : {j} : on prédit en moyenne {(1 - (j/l))*100}% mieux qu\'avec un modèle naïf')

# Filtrer les entrées où la RMSSE est définie (non None)
rmsse_filtrees = {k: v for k, v in dico_des_rmsse.items() if v is not None}

# S'il en reste, on affiche la meilleure
if rmsse_filtrees:
    meilleur_dataset = min(rmsse_filtrees, key=rmsse_filtrees.get)
    meilleure_rmsse = rmsse_filtrees[meilleur_dataset]
    print(f"Meilleure RMSSE : {meilleure_rmsse:.4f} pour le dataset {meilleur_dataset}")
else:
    print("⚠️ Aucune RMSSE valide disponible pour les séries analysées.")

print(f'RMSSE moyenne : {j/len(datasets_groupes_additionnels)}')
nb_rmsse_egales_1 = sum(1 for v in dico_des_rmsse.values() if v == 1)
print(f'Proportion de séries totalement imprévisibles : {(nb_rmsse_egales_1/len(datasets_groupes_additionnels))*100} %')
print(f'{l} séries ont pu être traitées')
print(f"{len(series_temp_features[series_temp_features['id_methode'] == 'echec'])} échecs")
print(f"{len(series_temp_features[series_temp_features['id_methode'] == 'serie_raccourcie'])} séries raccourcies")
print(f"{len(series_temp_features[series_temp_features['id_methode'] == 'succes'])} succès")

print(series_temp_features)
print(df_previsions_parametres_deja_connus.head())
print(df_previsions_parametres_deja_connus.tail())

14 datasets encore en lice
dataset146visite_grp85 faite, plus que 13. RMSSE : 0.26645063383436035 (succes_immediat) MCMC : False, Surajustement : False, raccourcissement : None
dataset148_grp54 faite, plus que 12. RMSSE : 0.2595562262311918 (succes_immediat) MCMC : True, Surajustement : False, raccourcissement : None
dataset148_grp107 faite, plus que 11. RMSSE : 0.37622809268491 (succes_immediat) MCMC : True, Surajustement : False, raccourcissement : None
dataset150_grp35 faite, plus que 10. RMSSE : 0.20880770745663854 (succes_immediat) MCMC : False, Surajustement : False, raccourcissement : None
dataset150_grp68 faite, plus que 9. RMSSE : 0.5803112813905301 (succes_immediat) MCMC : False, Surajustement : False, raccourcissement : None
dataset156action1_grp5 faite, plus que 8. RMSSE : 0.5140113748065466 (serie_raccourcie_60) MCMC : True, Surajustement : False, raccourcissement : 60
dataset156action1_grp89 faite, plus que 7. RMSSE : 0.5352752359654497 (succes_immediat) MCMC : False, Sur

In [ ]:
resultats_previsions_directes = []

for nom in datasets_groupes.keys() :
    if nom in series_temp_features_notebook_balayage['nom_serie'].values :
        try :
            df_prophet = datasets_groupes[nom].copy()
            ligne_param = series_temp_features_notebook_balayage[series_temp_features_notebook_balayage['nom_serie'] == nom].iloc[0]

            prior_scale = ligne_param['changepoint_prior_scale']
            use_mcmc = bool(ligne_param['mcmc_utilise'])
            saisonnalite = ligne_param['saisonnalite']
            mode_saisonnalite = ligne_param['mode_saisonnalite'] if saisonnalite else None

            match = re.search(r'grp(\d+)', nom)
            groupe_id = int(match.group(1)) if match else None

            model = Prophet(
                seasonality_mode=mode_saisonnalite if saisonnalite else 'additive',
                yearly_seasonality=bool(saisonnalite),
                weekly_seasonality=False,
                daily_seasonality=False,
                changepoint_prior_scale=prior_scale,
                n_changepoints=int(np.floor(len(df_prophet)/3)),
                **({'mcmc_samples': 150} if use_mcmc else {}),
                uncertainty_samples=1000
            )

            if use_mcmc:
                model.fit(df_prophet, show_progress=False)
            else:
                model.fit(df_prophet)


            future = model.make_future_dataframe(periods=12, freq='MS')
            forecast = model.predict(future)

            forecast[['yhat', 'yhat_lower', 'yhat_upper']] = forecast[['yhat', 'yhat_lower', 'yhat_upper']].clip(lower=0)

            forecast_12 = forecast.tail(12).copy()
            forecast_12['dataset_nom'] = nom
            forecast_12['groupe_id'] = groupe_id
            forecast_12 = forecast_12[['ds', 'yhat', 'yhat_lower', 'yhat_upper', 'dataset_nom', 'groupe_id']]

            resultats_previsions_directes.append(forecast_12)

            print(f"✅ Prévision terminée pour {nom}")

        except Exception as e:
            print(f"❌ Erreur sur {nom} : {e}")
            continue

# Ajouter les résultats à df_previsions_parametres_deja_connus
if resultats_previsions_directes :
    df_nouvelles_previsions = pd.concat(resultats_previsions_directes, ignore_index=True)
    df_nouvelles_previsions = df_nouvelles_previsions.rename(columns={
        'dataset_nom': 'graphique_id',
        'groupe_id': 'groupe_id',
        'ds': 'gd_date',
        'yhat': 'gd_y',
        'yhat_lower': 'gd_inf',
        'yhat_upper': 'gd_sup'
    })
    df_previsions_parametres_deja_connus = pd.concat([df_previsions_parametres_deja_connus, df_nouvelles_previsions], ignore_index=True)
    print("✅ Nouvelles prévisions ajoutées à df_previsions_parametres_deja_connus.")
else:
    print("⚠️ Aucune nouvelle prévision n’a été ajoutée.")

✅ Prévision terminée pour dataset146visite_grp35
✅ Prévision terminée pour dataset146visite_grp38
✅ Prévision terminée pour dataset146visite_grp47
✅ Prévision terminée pour dataset146visite_grp50
✅ Prévision terminée pour dataset146visite_grp56
✅ Prévision terminée pour dataset146visite_grp67
✅ Prévision terminée pour dataset146visite_grp68
✅ Prévision terminée pour dataset146visite_grp75
✅ Prévision terminée pour dataset146visite_grp76
✅ Prévision terminée pour dataset146visite_grp79
✅ Prévision terminée pour dataset146visite_grp83
✅ Prévision terminée pour dataset146visite_grp86
✅ Prévision terminée pour dataset146visite_grp118
✅ Prévision terminée pour dataset146rdv_grp13
✅ Prévision terminée pour dataset146rdv_grp35
✅ Prévision terminée pour dataset146rdv_grp83
✅ Prévision terminée pour dataset146rdv_grp118
✅ Prévision terminée pour dataset146rdv_grp153
✅ Prévision terminée pour dataset147_grp13
✅ Prévision terminée pour dataset147_grp14
✅ Prévision terminée pour dataset147_grp35
✅

In [111]:
# Il nous faut maintenant une correspondance entre les numéros des graphiques dans Superset et les noms datasetX_grpX
# Attention, ici on autoincrémente les numéros des graphiques de prévision à partir du numéro 168 mais si on vient à ajouter 
# Dans les graphiques de base un nouveau graphique, son identifiant superset sera 206 (à chaque fois qu'on ajoute un graphique)
# dans Superset, son identifiant est : identifiant du dernier graphique créé + 1. 
# Donc ici, le prochain graphique de base sera le 206, et donc si on veut dans le même temps ajouter son graphique de
# prévisions, il faur=dra lui attribuer comme identifiant le numéro 207 car ce sera le graphique créé juste après 

previsions_nom = [
    'dataset146visite',
    'dataset146proposition',
    'dataset146rdv',
    'dataset147',
    'dataset148',
    'dataset149',
    'dataset150',
    'dataset151',
    'dataset152',
    'dataset153',
    'dataset154',
    'dataset155',
    'dataset156action1',
    'dataset156action2',
    'dataset156action3',
    'dataset156action4',
    'dataset156action5',
    'dataset157',
    'dataset158offres',
    'dataset158demandes',
    'dataset158mandats',
    'dataset159offres',
    'dataset159demandes',
    'dataset159mandats',
    'dataset160rejete',
    'dataset160npai',
    'dataset161delivres',
    'dataset161cliques',
    'dataset161ouverts',
    'dataset161rejetes',
    'dataset164simple',
    'dataset164demande',
    'dataset164exclusif',
    'dataset164coexclusif',
    'dataset164triexclusif',
    'dataset164preferentiel',
    'dataset164delegation',
    'dataset164quadriexclusif'
]

# On auto-incrémente les identifiants à partir de 168 car jusque 166, c'est les graphiques des données historiques, 
# ensuite on a le 167 qui était un graphique de test
# Et donc nos graphiques de prévisions auront des identifiants qui commenceront à 168

df_ids = pd.DataFrame({
    'nom': previsions_nom,
    'id': list(range(168, 168 + len(previsions_nom)))
})

In [ ]:
# On rouvre la connexion pyodbc à SQL Server

conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=172.16.11.34;"
    "DATABASE=Superset;"
    "UID=quentin;"
    "PWD={Barbidur1;SQL}"
)

conn = pyodbc.connect(conn_str)
cursor = conn.cursor()

for _, row in df_ids.iterrows():
    nom = row['nom']
    id = row['id']
    cursor.execute(
        "INSERT INTO previsions_graphiques_noms (previsions_graphique_id, previsions_graphique_nom) VALUES (?, ?)",
        (id, nom)
    )
conn.commit()

In [113]:
# On remplace les noms datasetX..._grpX par des noms de la forme : datasetX (pas besoin de savoir le groupe dans le nom,
# de toute facon il y a dejà une colonne groupe_id et on filtrera une fois dans Superset)

# ATTENTION ! ON IDENTIFIE ICI LES GRAPHIQUES DE PREVISIONS, PAS LES GRAPHIQUES HISTORIQUES AUXQUELS SONT RELIEES LES PREVISIONS !

# On enlève les _grpX des noms dans df_previsions_parametres_deja_connus
df_previsions_parametres_deja_connus['graphique_id'] = df_previsions_parametres_deja_connus['graphique_id'].str.replace(r'_grp\d+$', '', regex=True)

# On crée un dico de correspondance avec df_ids
mapping_ids = df_ids.set_index('nom')['id']

# On remplace les noms par les vrais identifiants
df_previsions_parametres_deja_connus['graphique_id'] = df_previsions_parametres_deja_connus['graphique_id'].map(mapping_ids)

print(df_previsions_parametres_deja_connus)

        gd_date      gd_y    gd_inf     gd_sup  graphique_id  groupe_id
0    2025-07-01  7.410840  0.000000  16.571215           168         85
1    2025-08-01  7.328977  0.000000  16.284825           168         85
2    2025-09-01  7.247115  0.000000  17.762301           168         85
3    2025-10-01  7.167893  0.000000  16.926867           168         85
4    2025-11-01  7.086030  0.000000  16.821668           168         85
...         ...       ...       ...        ...           ...        ...
7459 2026-05-01  5.176219  4.160962   6.098978           205        133
7460 2026-06-01  3.982953  3.003931   4.922807           205        133
7461 2026-07-01  3.474772  2.529127   4.436295           205        133
7462 2026-08-01  3.499955  2.510643   4.461317           205        133
7463 2026-09-01  3.323782  2.375429   4.280428           205        133

[7464 rows x 6 columns]


In [114]:
df_previsions_parametres_deja_connus.to_excel("df_previsions_parametres_deja_connus.xlsx", index=False)

In [ ]:
# Compter deux minutes d'exécution

df_previsions_parametres_deja_connus = df_previsions_parametres_deja_connus.rename(columns={
    'gd_date' : 'prevision_date',
    'gd_y' : 'yhat',
    'gd_inf' : 'yhat_inf',
    'gd_sup' : 'yhat_sup',
    'graphique_id' : 'previsions_graphique_id'
})

insert_sql = """
    INSERT INTO previsions_graphiques (previsions_graphique_id, groupe_id, prevision_date, yhat, yhat_inf, yhat_sup)
    VALUES (?, ?, ?, ?, ?, ?)
"""

print("Insertion des nouvelles données...")

# Parcours des lignes du DataFrame
for _, row in df_previsions_parametres_deja_connus.iterrows() :
    cursor.execute(insert_sql, (
        int(row['previsions_graphique_id']),
        int(row['groupe_id']),
        row['prevision_date'],
        float(row['yhat']),
        float(row['yhat_inf']),
        float(row['yhat_sup'])
    ))

# On ferme la connexion

conn.commit()
cursor.close()
conn.close()


print("Données insérées avec succès.")

Insertion des nouvelles données...
Données insérées avec succès.
